In [5]:
# =============================================================================
# NOTEBOOK 17 — CELL 1
# XAI SAFETY + FROZEN MODEL AUDIT
# FINAL CORRECTED VERSION — NO METRICS-FILE DEPENDENCY
# =============================================================================

from pathlib import Path
import json
import pandas as pd


# =============================================================================
# 1. ROOT PATHS
# =============================================================================

PROJECT_ROOT = Path(r"C:\TBP\Metadata")

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

NOTEBOOK17_ROOT = (
    FINAL_ROOT
    / "Notebook17_Final_Explainability_Analysis"
)

CELL1_ROOT = (
    NOTEBOOK17_ROOT
    / "Cell1_XAI_Safety_Frozen_Model_Audit"
)

CELL1_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("NOTEBOOK 17 — CELL 1")
print("XAI SAFETY + FROZEN MODEL AUDIT")
print("=" * 90)


# =============================================================================
# 2. FROZEN COHORT
# =============================================================================

FROZEN_COHORT_FILE = (
    FINAL_ROOT
    / "Notebook07_Cell24F_FINAL_CLEAN_PSPNet_CXR_COHORT.csv"
)

if not FROZEN_COHORT_FILE.exists():
    raise FileNotFoundError(
        f"Frozen cohort not found:\n{FROZEN_COHORT_FILE}"
    )

frozen_df = pd.read_csv(
    FROZEN_COHORT_FILE
)

print("\nFrozen cohort:")
print(FROZEN_COHORT_FILE)
print("Rows:", len(frozen_df))


# =============================================================================
# 3. COLUMN RESOLUTION
# =============================================================================

def find_column(df, candidates, description):

    for col in candidates:
        if col in df.columns:
            return col

    raise KeyError(
        f"Could not find {description}.\n"
        f"Available columns:\n{list(df.columns)}"
    )


condition_col = find_column(
    frozen_df,
    [
        "condition_id",
        "Condition_ID",
        "condition"
    ],
    "condition ID"
)

target_col = find_column(
    frozen_df,
    [
        "target_binary",
        "target",
        "target_label"
    ],
    "target"
)


# =============================================================================
# 4. FROZEN COHORT AUDIT
# =============================================================================

print("\n" + "=" * 90)
print("FROZEN COHORT AUDIT")
print("=" * 90)

if len(frozen_df) != 1976:
    raise RuntimeError(
        f"Expected 1976 conditions, found {len(frozen_df)}."
    )

if frozen_df[condition_col].nunique() != 1976:
    raise RuntimeError(
        "Condition IDs are not unique."
    )

targets = frozen_df[
    target_col
].astype(int)

if set(targets.unique()) != {0, 1}:
    raise RuntimeError(
        "Target must contain exactly DS=0 and DR=1."
    )

ds_count = int(
    (targets == 0).sum()
)

dr_count = int(
    (targets == 1).sum()
)

if ds_count != 704:
    raise RuntimeError(
        f"Expected DS=704, found {ds_count}."
    )

if dr_count != 1272:
    raise RuntimeError(
        f"Expected DR=1272, found {dr_count}."
    )

print("Total conditions     :", 1976)
print("Unique conditions    :", 1976)
print("DS conditions        :", ds_count)
print("DR conditions        :", dr_count)
print("Target encoding      : DS=0, DR=1")
print("Cohort audit         : PASS")


# =============================================================================
# 5. AUTHORITATIVE FROZEN SPLIT
# =============================================================================

AUTHORITATIVE_SPLIT_FILE = (
    FINAL_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
    / "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

if not AUTHORITATIVE_SPLIT_FILE.exists():
    raise FileNotFoundError(
        "Authoritative split file not found:\n"
        f"{AUTHORITATIVE_SPLIT_FILE}"
    )

split_df = pd.read_csv(
    AUTHORITATIVE_SPLIT_FILE
)

print("\n" + "=" * 90)
print("AUTHORITATIVE SPLIT AUDIT")
print("=" * 90)

if len(split_df) != 1976:
    raise RuntimeError(
        f"Expected 1976 split rows, found {len(split_df)}."
    )


split_condition_col = find_column(
    split_df,
    [
        "condition_id",
        "Condition_ID",
        "condition"
    ],
    "split condition ID"
)

split_col = find_column(
    split_df,
    [
        "split",
        "dataset_split"
    ],
    "split"
)

split_target_col = find_column(
    split_df,
    [
        "target_binary",
        "target",
        "target_label"
    ],
    "split target"
)


# =============================================================================
# 6. SPLIT NORMALIZATION
# =============================================================================

split_values = (
    split_df[
        split_col
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

split_targets = (
    split_df[
        split_target_col
    ]
    .astype(int)
)

train_mask = split_values.isin(
    [
        "train",
        "training"
    ]
)

val_mask = split_values.isin(
    [
        "validation",
        "val",
        "valid"
    ]
)

test_mask = split_values.isin(
    [
        "test",
        "testing"
    ]
)

train_count = int(
    train_mask.sum()
)

val_count = int(
    val_mask.sum()
)

test_count = int(
    test_mask.sum()
)

if train_count != 1185:
    raise RuntimeError(
        f"Expected train=1185, found {train_count}."
    )

if val_count != 395:
    raise RuntimeError(
        f"Expected validation=395, found {val_count}."
    )

if test_count != 396:
    raise RuntimeError(
        f"Expected test=396, found {test_count}."
    )

print("Train                   :", train_count)
print("Validation              :", val_count)
print("Test                    :", test_count)


# =============================================================================
# 7. SPLIT × TARGET AUDIT
# =============================================================================

expected_distribution = {

    ("train", 0): 422,
    ("train", 1): 763,

    ("validation", 0): 141,
    ("validation", 1): 254,

    ("test", 0): 141,
    ("test", 1): 255,
}

actual_distribution = {

    ("train", 0):
        int(
            (
                train_mask
                &
                (split_targets == 0)
            ).sum()
        ),

    ("train", 1):
        int(
            (
                train_mask
                &
                (split_targets == 1)
            ).sum()
        ),

    ("validation", 0):
        int(
            (
                val_mask
                &
                (split_targets == 0)
            ).sum()
        ),

    ("validation", 1):
        int(
            (
                val_mask
                &
                (split_targets == 1)
            ).sum()
        ),

    ("test", 0):
        int(
            (
                test_mask
                &
                (split_targets == 0)
            ).sum()
        ),

    ("test", 1):
        int(
            (
                test_mask
                &
                (split_targets == 1)
            ).sum()
        ),
}

if actual_distribution != expected_distribution:
    raise RuntimeError(
        "Split × target distribution mismatch.\n"
        f"Expected: {expected_distribution}\n"
        f"Actual: {actual_distribution}"
    )

print(
    f"Train : DS={actual_distribution[('train',0)]}, "
    f"DR={actual_distribution[('train',1)]}"
)

print(
    f"Val   : DS={actual_distribution[('validation',0)]}, "
    f"DR={actual_distribution[('validation',1)]}"
)

print(
    f"Test  : DS={actual_distribution[('test',0)]}, "
    f"DR={actual_distribution[('test',1)]}"
)

print("Split × target audit    : PASS")


# =============================================================================
# 8. CONDITION OVERLAP
# =============================================================================

train_ids = set(
    split_df.loc[
        train_mask,
        split_condition_col
    ].astype(str)
)

val_ids = set(
    split_df.loc[
        val_mask,
        split_condition_col
    ].astype(str)
)

test_ids = set(
    split_df.loc[
        test_mask,
        split_condition_col
    ].astype(str)
)

if train_ids & val_ids:
    raise RuntimeError(
        "Train/validation condition overlap detected."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "Train/test condition overlap detected."
    )

if val_ids & test_ids:
    raise RuntimeError(
        "Validation/test condition overlap detected."
    )

print("Condition overlap       : NONE")


# =============================================================================
# 9. FROZEN CHECKPOINTS
# =============================================================================

CXR_CHECKPOINT = (
    FINAL_ROOT
    / "CoAtNet_Training_Run2_10C"
    / "checkpoints"
    / "Notebook10C_Run2_Best_Validation_ROC_AUC.pth"
)

GENOMIC_CHECKPOINT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell7_Transformer_Training"
    / "Notebook13_Cell7_Best_Validation_ROC_AUC.pth"
)

MULTIMODAL_CHECKPOINT = (
    FINAL_ROOT
    / "Multimodal_15"
    / "Cell4_Fusion_Training"
    / "checkpoints"
    / "Notebook15_Cell4_Best_Multimodal_Validation_ROC_AUC.pth"
)

checkpoint_paths = {

    "CXR Run-2":
        CXR_CHECKPOINT,

    "Genomic":
        GENOMIC_CHECKPOINT,

    "CXR + Genomic":
        MULTIMODAL_CHECKPOINT,

}

print("\n" + "=" * 90)
print("FROZEN CHECKPOINT AUDIT")
print("=" * 90)

for name, path in checkpoint_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} checkpoint not found:\n{path}"
        )

    print(
        f"{name:<20}: PASS "
        f"({path.stat().st_size / (1024**2):.2f} MB)"
    )


# =============================================================================
# 10. FROZEN TEST PREDICTIONS
# =============================================================================

CXR_TEST_PREDICTIONS = (
    FINAL_ROOT
    / "CoAtNet_Test_Evaluation_Run2_10G"
    / "Cell2_Final_Test_Predictions"
    / "Notebook10G_Cell2_Run2_FINAL_396_Test_Predictions.csv"
)

GENOMIC_TEST_PREDICTIONS = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell8_Frozen_Test_Evaluation"
    / "Notebook13_Cell8_FINAL_Frozen_Test_Predictions.csv"
)

MULTIMODAL_TEST_PREDICTIONS = (
    FINAL_ROOT
    / "Multimodal_15"
    / "Cell6_Frozen_Test_Evaluation"
    / "Notebook15_Cell6_FINAL_Frozen_Test_Predictions.csv"
)

prediction_paths = {

    "CXR":
        CXR_TEST_PREDICTIONS,

    "Genomic":
        GENOMIC_TEST_PREDICTIONS,

    "CXR + Genomic":
        MULTIMODAL_TEST_PREDICTIONS,

}

print("\n" + "=" * 90)
print("FROZEN TEST PREDICTION AUDIT")
print("=" * 90)

prediction_dfs = {}

for name, path in prediction_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} predictions not found:\n{path}"
        )

    df = pd.read_csv(
        path
    )

    if len(df) != 396:
        raise RuntimeError(
            f"{name}: expected 396 predictions, "
            f"found {len(df)}."
        )

    prediction_dfs[name] = df

    print(
        f"{name:<20}: PASS "
        f"({len(df)} predictions)"
    )


# =============================================================================
# 11. TEST CONDITION ALIGNMENT
# =============================================================================

def get_condition_column(df):

    for col in [
        "condition_id",
        "Condition_ID",
        "condition"
    ]:

        if col in df.columns:
            return col

    raise KeyError(
        "Condition ID column not found:\n"
        f"{list(df.columns)}"
    )


print("\n" + "=" * 90)
print("TEST CONDITION ALIGNMENT")
print("=" * 90)

for name, df in prediction_dfs.items():

    col = get_condition_column(
        df
    )

    prediction_ids = set(
        df[col].astype(str)
    )

    if prediction_ids != test_ids:

        raise RuntimeError(
            f"{name}: prediction IDs do not exactly "
            f"match the frozen test cohort."
        )

    print(
        f"{name:<20}: EXACT 396/396 MATCH"
    )


# =============================================================================
# 12. TEST TARGET ALIGNMENT
# =============================================================================

target_map = dict(
    zip(
        split_df[
            split_condition_col
        ].astype(str),

        split_df[
            split_target_col
        ].astype(int)
    )
)

print("\n" + "=" * 90)
print("TEST TARGET ALIGNMENT")
print("=" * 90)

for name, df in prediction_dfs.items():

    col = get_condition_column(
        df
    )

    if "target_binary" not in df.columns:

        raise RuntimeError(
            f"{name}: target_binary column missing."
        )

    mismatches = 0

    for _, row in df.iterrows():

        condition_id = str(
            row[col]
        )

        prediction_target = int(
            row["target_binary"]
        )

        if (
            prediction_target
            !=
            target_map[
                condition_id
            ]
        ):

            mismatches += 1

    if mismatches != 0:

        raise RuntimeError(
            f"{name}: {mismatches} target mismatches."
        )

    print(
        f"{name:<20}: PASS"
    )


# =============================================================================
# 13. XAI SAFETY DECLARATION
# =============================================================================

xai_safety = {

    "training_performed":
        False,

    "fine_tuning_performed":
        False,

    "checkpoint_selection_performed":
        False,

    "threshold_tuning_performed":
        False,

    "test_set_used_for_model_selection":
        False,

    "test_set_used_for_threshold_selection":
        False,

    "test_predictions_modified":
        False,

    "frozen_cohort_modified":
        False,

}


# =============================================================================
# 14. SAVE AUDIT SUMMARY
# =============================================================================

AUDIT_SUMMARY_FILE = (
    CELL1_ROOT
    / "Notebook17_Cell1_XAI_Safety_Frozen_Model_Audit_Summary.json"
)

audit_output = {

    "status":
        "PASS",

    "frozen_cohort_file":
        str(FROZEN_COHORT_FILE),

    "authoritative_split_file":
        str(AUTHORITATIVE_SPLIT_FILE),

    "frozen_cohort": {

        "conditions":
            1976,

        "train":
            1185,

        "validation":
            395,

        "test":
            396,

        "DS":
            704,

        "DR":
            1272,

        "test_DS":
            141,

        "test_DR":
            255,

    },

    "split_target_distribution":
        {
            str(k): v
            for k, v
            in actual_distribution.items()
        },

    "checkpoints": {
        key: str(value)
        for key, value
        in checkpoint_paths.items()
    },

    "test_predictions": {
        key: str(value)
        for key, value
        in prediction_paths.items()
    },

    "xai_safety":
        xai_safety,

    "next_step":
        "Notebook17 Cell2 — Frozen CoAtNet CXR Grad-CAM analysis",

}

with open(
    AUDIT_SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_output,
        f,
        indent=4
    )


# =============================================================================
# 15. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 90)
print("NOTEBOOK 17 — CELL 1 STATUS: PASS")
print("=" * 90)

print("Frozen cohort           : 1976")
print("Train / Val / Test      : 1185 / 395 / 396")
print("Cohort DS / DR          : 704 / 1272")
print("Test DS / DR            : 141 / 255")

print("\nCXR checkpoint          : PASS")
print("Genomic checkpoint      : PASS")
print("Multimodal checkpoint   : PASS")

print("\nCXR test predictions    : PASS")
print("Genomic predictions     : PASS")
print("Multimodal predictions  : PASS")

print("\nTraining performed          : NO")
print("Fine-tuning performed      : NO")
print("Threshold tuning           : NO")
print("Test-set model selection   : NO")
print("Frozen data modified       : NO")

print("\nAudit summary:")
print(AUDIT_SUMMARY_FILE)

print("\nCELL 1 COMPLETE.")
print("Proceed to Notebook 17 — Cell 2.")

NOTEBOOK 17 — CELL 1
XAI SAFETY + FROZEN MODEL AUDIT

Frozen cohort:
C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook07_Cell24F_FINAL_CLEAN_PSPNet_CXR_COHORT.csv
Rows: 1976

FROZEN COHORT AUDIT
Total conditions     : 1976
Unique conditions    : 1976
DS conditions        : 704
DR conditions        : 1272
Target encoding      : DS=0, DR=1
Cohort audit         : PASS

AUTHORITATIVE SPLIT AUDIT
Train                   : 1185
Validation              : 395
Test                    : 396
Train : DS=422, DR=763
Val   : DS=141, DR=254
Test  : DS=141, DR=255
Split × target audit    : PASS
Condition overlap       : NONE

FROZEN CHECKPOINT AUDIT
CXR Run-2           : PASS (305.50 MB)
Genomic             : PASS (3.30 MB)
CXR + Genomic       : PASS (3.03 MB)

FROZEN TEST PREDICTION AUDIT
CXR                 : PASS (396 predictions)
Genomic             : PASS (396 predictions)
CXR + Genomic       : PASS (396 predictions)

TEST CONDITION AL

In [6]:
# =============================================================================
# NOTEBOOK 17 — CELL 2
# FROZEN CoAtNet CXR Grad-CAM EXPLAINABILITY
# =============================================================================
#
# PURPOSE
# -------
# Generate Grad-CAM explanations for the FINAL FROZEN CXR Run-2 CoAtNet model
# on all 396 frozen test conditions.
#
# IMPORTANT
# ----------
# This cell:
#   - uses the already-frozen CXR checkpoint
#   - uses the already-frozen 224x224 NPY inputs
#   - uses ONLY the authoritative frozen test IDs
#   - does NOT train
#   - does NOT fine-tune
#   - does NOT select a checkpoint
#   - does NOT tune a threshold
#   - does NOT modify test predictions
#
# Additionally, the loaded checkpoint predictions are independently compared
# against the official frozen Notebook10G Run-2 predictions before Grad-CAM
# maps are generated.
#
# OUTPUTS
# -------
#   1. 396 Grad-CAM maps (.npy)
#   2. 396 Grad-CAM overlay images (.png)
#   3. Grad-CAM metadata CSV
#   4. Prediction consistency audit
#   5. Cell summary JSON
# =============================================================================

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import timm


warnings.filterwarnings("ignore")


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_ROOT = Path(r"C:\TBP\Metadata")

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

NOTEBOOK17_ROOT = (
    FINAL_ROOT
    / "Notebook17_Final_Explainability_Analysis"
)

CELL1_ROOT = (
    NOTEBOOK17_ROOT
    / "Cell1_XAI_Safety_Frozen_Model_Audit"
)

CELL2_ROOT = (
    NOTEBOOK17_ROOT
    / "Cell2_CXR_GradCAM"
)

CAM_NPY_ROOT = (
    CELL2_ROOT
    / "GradCAM_Maps_NPY"
)

CAM_PNG_ROOT = (
    CELL2_ROOT
    / "GradCAM_Overlays_PNG"
)

CELL2_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CAM_NPY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CAM_PNG_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. AUTHORITATIVE FILES
# =============================================================================

CXR_CHECKPOINT = (
    FINAL_ROOT
    / "CoAtNet_Training_Run2_10C"
    / "checkpoints"
    / "Notebook10C_Run2_Best_Validation_ROC_AUC.pth"
)

CXR_TEST_PREDICTIONS = (
    FINAL_ROOT
    / "CoAtNet_Test_Evaluation_Run2_10G"
    / "Cell2_Final_Test_Predictions"
    / "Notebook10G_Cell2_Run2_FINAL_396_Test_Predictions.csv"
)

AUTHORITATIVE_SPLIT_FILE = (
    FINAL_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
    / "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

CXR_INPUT_ROOT = (
    FINAL_ROOT
    / "CoAtNet_224_Final_Input"
)

CXR_NPY_ROOT = (
    CXR_INPUT_ROOT
    / "NPY"
)

CXR_MANIFEST_FILE = (
    CXR_INPUT_ROOT
    / "QC"
    / "Notebook08_Cell26B_FINAL_1976_CoAtNet_Input_Manifest.csv"
)

CELL1_AUDIT_FILE = (
    CELL1_ROOT
    / "Notebook17_Cell1_XAI_Safety_Frozen_Model_Audit_Summary.json"
)


# =============================================================================
# 3. PATH GATE
# =============================================================================

print("=" * 90)
print("NOTEBOOK 17 — CELL 2")
print("FROZEN CoAtNet CXR Grad-CAM EXPLAINABILITY")
print("=" * 90)

required_paths = {

    "CXR checkpoint":
        CXR_CHECKPOINT,

    "CXR frozen predictions":
        CXR_TEST_PREDICTIONS,

    "Authoritative split":
        AUTHORITATIVE_SPLIT_FILE,

    "CXR input root":
        CXR_INPUT_ROOT,

    "CXR NPY root":
        CXR_NPY_ROOT,

    "CXR manifest":
        CXR_MANIFEST_FILE,

    "Cell 1 audit":
        CELL1_AUDIT_FILE,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<28}: PASS"
    )


# =============================================================================
# 4. CELL 1 SAFETY GATE
# =============================================================================

with open(
    CELL1_AUDIT_FILE,
    "r",
    encoding="utf-8"
) as f:

    cell1_audit = json.load(f)


if cell1_audit.get("status") != "PASS":

    raise RuntimeError(
        "Notebook17 Cell1 did not finish with PASS."
    )


xai_safety = cell1_audit.get(
    "xai_safety",
    {}
)

for key, value in xai_safety.items():

    if value is not False:

        raise RuntimeError(
            f"Cell1 XAI safety gate failed: "
            f"{key}={value}"
        )

print(
    "\nCell 1 safety gate: PASS"
)


# =============================================================================
# 5. LOAD AUTHORITATIVE SPLIT
# =============================================================================

split_df = pd.read_csv(
    AUTHORITATIVE_SPLIT_FILE
)


def find_column(
    df,
    candidates,
    description
):

    for col in candidates:

        if col in df.columns:

            return col

    raise KeyError(
        f"Could not find {description}.\n"
        f"Available columns:\n{list(df.columns)}"
    )


split_condition_col = find_column(
    split_df,
    [
        "condition_id",
        "Condition_ID",
        "condition"
    ],
    "condition ID"
)

split_col = find_column(
    split_df,
    [
        "split",
        "dataset_split"
    ],
    "split"
)

split_target_col = find_column(
    split_df,
    [
        "target_binary",
        "target",
        "target_label"
    ],
    "target"
)

split_values = (
    split_df[
        split_col
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

test_mask = split_values.isin(
    [
        "test",
        "testing"
    ]
)

authoritative_test_df = (
    split_df.loc[
        test_mask
    ].copy()
)

if len(authoritative_test_df) != 396:

    raise RuntimeError(
        f"Expected 396 authoritative test conditions, "
        f"found {len(authoritative_test_df)}."
    )

authoritative_test_df[
    split_condition_col
] = authoritative_test_df[
    split_condition_col
].astype(str)

authoritative_test_df[
    split_target_col
] = authoritative_test_df[
    split_target_col
].astype(int)

print(
    "Authoritative test conditions:",
    len(authoritative_test_df)
)


# =============================================================================
# 6. LOAD OFFICIAL FROZEN CXR PREDICTIONS
# =============================================================================

official_predictions = pd.read_csv(
    CXR_TEST_PREDICTIONS
)

if len(official_predictions) != 396:

    raise RuntimeError(
        f"Expected 396 official CXR predictions, "
        f"found {len(official_predictions)}."
    )

prediction_condition_col = find_column(
    official_predictions,
    [
        "condition_id",
        "Condition_ID",
        "condition"
    ],
    "prediction condition ID"
)

if "target_binary" not in official_predictions.columns:

    raise RuntimeError(
        "Official CXR prediction file does not contain "
        "'target_binary'."
    )

if "probability_DR" not in official_predictions.columns:

    raise RuntimeError(
        "Official CXR prediction file does not contain "
        "'probability_DR'."
    )

if "predicted_binary" not in official_predictions.columns:

    raise RuntimeError(
        "Official CXR prediction file does not contain "
        "'predicted_binary'."
    )

official_predictions[
    prediction_condition_col
] = official_predictions[
    prediction_condition_col
].astype(str)

official_test_ids = set(
    official_predictions[
        prediction_condition_col
    ]
)

authoritative_test_ids = set(
    authoritative_test_df[
        split_condition_col
    ]
)

if official_test_ids != authoritative_test_ids:

    raise RuntimeError(
        "Official CXR prediction IDs do not match "
        "the authoritative frozen test cohort."
    )

print(
    "Official CXR predictions: 396/396 aligned"
)


# =============================================================================
# 7. LOAD CoAtNet INPUT MANIFEST
# =============================================================================

manifest_df = pd.read_csv(
    CXR_MANIFEST_FILE
)

print(
    "\nManifest rows:",
    len(manifest_df)
)


manifest_condition_col = find_column(
    manifest_df,
    [
        "condition_id",
        "Condition_ID",
        "condition"
    ],
    "manifest condition ID"
)

manifest_df[
    manifest_condition_col
] = manifest_df[
    manifest_condition_col
].astype(str)


# =============================================================================
# 8. RESOLVE NPY PATH COLUMN
# =============================================================================

npy_path_column = None

npy_candidates = [

    "npy_path",
    "NPY_path",
    "npy_file",
    "npy_filename",
    "filename",
    "file_name",
    "output_path",
    "input_path",
    "CoAtNet_NPY",
]

for col in npy_candidates:

    if col in manifest_df.columns:

        npy_path_column = col
        break


def resolve_npy_path(
    condition_id,
    manifest_row=None
):

    # -------------------------------------------------------------------------
    # First: use manifest path if available.
    # -------------------------------------------------------------------------

    if (
        manifest_row is not None
        and
        npy_path_column is not None
    ):

        raw_value = manifest_row[
            npy_path_column
        ]

        if pd.notna(raw_value):

            raw_path = Path(
                str(raw_value)
            )

            candidates = [

                raw_path,

                CXR_NPY_ROOT
                / raw_path,

                CXR_INPUT_ROOT
                / raw_path,

            ]

            for candidate in candidates:

                if candidate.exists():

                    return candidate


    # -------------------------------------------------------------------------
    # Second: search NPY directory using condition ID.
    # -------------------------------------------------------------------------

    matches = list(
        CXR_NPY_ROOT.rglob(
            f"*{condition_id}*.npy"
        )
    )

    if len(matches) == 1:

        return matches[0]

    if len(matches) > 1:

        # Prefer exact stem match.
        exact = [
            p for p in matches
            if p.stem == condition_id
        ]

        if len(exact) == 1:

            return exact[0]

        raise RuntimeError(
            f"Multiple NPY files found for condition "
            f"{condition_id}:\n"
            + "\n".join(
                str(p) for p in matches
            )
        )

    raise FileNotFoundError(
        f"No CoAtNet NPY found for condition "
        f"{condition_id}"
    )


# =============================================================================
# 9. BUILD TEST INPUT TABLE
# =============================================================================

print("\nResolving frozen test NPY inputs...")

manifest_lookup = {}

for _, row in manifest_df.iterrows():

    cid = str(
        row[
            manifest_condition_col
        ]
    )

    manifest_lookup[cid] = row


test_records = []

for _, row in authoritative_test_df.iterrows():

    cid = str(
        row[
            split_condition_col
        ]
    )

    if cid not in manifest_lookup:

        raise RuntimeError(
            f"Condition {cid} missing from "
            f"CoAtNet input manifest."
        )

    npy_path = resolve_npy_path(
        cid,
        manifest_lookup[cid]
    )

    test_records.append({

        "condition_id":
            cid,

        "target_binary":
            int(
                row[
                    split_target_col
                ]
            ),

        "npy_path":
            str(npy_path),

    })


test_input_df = pd.DataFrame(
    test_records
)


if len(test_input_df) != 396:

    raise RuntimeError(
        "Failed to construct 396 test input records."
    )

if test_input_df[
    "condition_id"
].nunique() != 396:

    raise RuntimeError(
        "Test input condition IDs are not unique."
    )

print(
    "Resolved test NPY inputs:",
    len(test_input_df)
)


# =============================================================================
# 10. VERIFY ALL NPY FILES
# =============================================================================

for path in test_input_df[
    "npy_path"
]:

    if not Path(path).exists():

        raise FileNotFoundError(
            f"NPY file missing:\n{path}"
        )

print(
    "All 396 NPY files: PASS"
)


# =============================================================================
# 11. LOAD CoAtNet
# =============================================================================

print("\n" + "=" * 90)
print("LOADING FROZEN CoAtNet RUN-2")
print("=" * 90)

device = torch.device(
    "cpu"
)

torch.set_grad_enabled(True)

model = timm.create_model(
    "coatnet_0_rw_224",
    pretrained=False,
    num_classes=2
)


# =============================================================================
# 12. ROBUST CHECKPOINT LOADER
# =============================================================================

checkpoint = torch.load(
    CXR_CHECKPOINT,
    map_location="cpu"
)

if isinstance(
    checkpoint,
    dict
):

    possible_keys = [

        "state_dict",

        "model_state_dict",

        "model",

        "net",

        "weights",

    ]

    state_dict = None

    for key in possible_keys:

        if key in checkpoint:

            candidate = checkpoint[
                key
            ]

            if isinstance(
                candidate,
                dict
            ):

                state_dict = candidate
                break

    if state_dict is None:

        # Direct state dictionary.
        if all(
            isinstance(k, str)
            for k in checkpoint.keys()
        ):

            state_dict = checkpoint

else:

    state_dict = checkpoint


if state_dict is None:

    raise RuntimeError(
        "Could not extract state_dict from checkpoint."
    )


# Remove DataParallel prefix if present.

clean_state_dict = {}

for key, value in state_dict.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):

        new_key = new_key[
            len("module.") :
        ]

    clean_state_dict[
        new_key
    ] = value


load_result = model.load_state_dict(
    clean_state_dict,
    strict=False
)


if load_result.missing_keys:

    raise RuntimeError(
        "Checkpoint has missing model keys:\n"
        + "\n".join(
            load_result.missing_keys
        )
    )

if load_result.unexpected_keys:

    raise RuntimeError(
        "Checkpoint has unexpected model keys:\n"
        + "\n".join(
            load_result.unexpected_keys
        )
    )


model = model.to(
    device
)

model.eval()

print(
    "Checkpoint loaded: PASS"
)


# =============================================================================
# 13. FIND FINAL CONVOLUTIONAL TARGET LAYER
# =============================================================================

conv_layers = []

for name, module in model.named_modules():

    if isinstance(
        module,
        nn.Conv2d
    ):

        conv_layers.append(
            (
                name,
                module
            )
        )


if len(conv_layers) == 0:

    raise RuntimeError(
        "No Conv2d layer found in CoAtNet."
    )


target_layer_name, target_layer = (
    conv_layers[-1]
)

print(
    "\nGrad-CAM target layer:"
)

print(
    target_layer_name
)

print(
    target_layer
)


# =============================================================================
# 14. IMAGE PREPROCESSING
# =============================================================================
#
# Notebook10C Run-2 preprocessing:
#
#   stored NPY:
#       RGB float32
#       range [-1,1]
#
#   reconstruction:
#       [-1,1] -> [0,1]
#
#   ImageNet normalization:
#       mean = [0.485,0.456,0.406]
#       std  = [0.229,0.224,0.225]
#
# No augmentation is used for explainability.
# =============================================================================

IMAGENET_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406
    ],
    dtype=torch.float32
).view(
    3, 1, 1
)

IMAGENET_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225
    ],
    dtype=torch.float32
).view(
    3, 1, 1
)


def load_cxr_tensor(
    npy_path
):

    array = np.load(
        npy_path
    )

    if array.shape != (
        224,
        224,
        3
    ):

        raise RuntimeError(
            f"Unexpected NPY shape "
            f"{array.shape} for {npy_path}"
        )

    array = array.astype(
        np.float32
    )

    if not np.isfinite(
        array
    ).all():

        raise RuntimeError(
            f"Non-finite values in {npy_path}"
        )

    # Frozen stored representation [-1,1].
    image_01 = (
        array + 1.0
    ) / 2.0

    image_01 = np.clip(
        image_01,
        0.0,
        1.0
    )

    tensor = torch.from_numpy(
        image_01
    ).permute(
        2,
        0,
        1
    ).float()

    tensor = (
        tensor
        -
        IMAGENET_MEAN
    ) / IMAGENET_STD

    return tensor.unsqueeze(
        0
    )


def tensor_to_display_image(
    tensor
):

    # tensor is normalized ImageNet tensor.
    image = (
        tensor.squeeze(0).detach().cpu()
        * IMAGENET_STD
        +
        IMAGENET_MEAN
    )

    image = torch.clamp(
        image,
        0.0,
        1.0
    )

    image = (
        image.permute(
            1,
            2,
            0
        )
        .numpy()
    )

    return image


# =============================================================================
# 15. GRAD-CAM IMPLEMENTATION
# =============================================================================

class GradCAM:

    def __init__(
        self,
        model,
        target_layer
    ):

        self.model = model

        self.target_layer = target_layer

        self.activations = None

        self.gradients = None

        self.forward_handle = (
            self.target_layer.register_forward_hook(
                self._forward_hook
            )
        )

        self.backward_handle = (
            self.target_layer.register_full_backward_hook(
                self._backward_hook
            )
        )


    def _forward_hook(
        self,
        module,
        inputs,
        output
    ):

        self.activations = output


    def _backward_hook(
        self,
        module,
        grad_input,
        grad_output
    ):

        self.gradients = grad_output[0]


    def remove_hooks(
        self
    ):

        self.forward_handle.remove()

        self.backward_handle.remove()


    def generate(
        self,
        input_tensor,
        target_class
    ):

        self.model.zero_grad(
            set_to_none=True
        )

        self.activations = None

        self.gradients = None

        output = self.model(
            input_tensor
        )

        score = output[
            0,
            target_class
        ]

        score.backward(
            retain_graph=False
        )

        if self.activations is None:

            raise RuntimeError(
                "Grad-CAM activations were not captured."
            )

        if self.gradients is None:

            raise RuntimeError(
                "Grad-CAM gradients were not captured."
            )

        activations = self.activations

        gradients = self.gradients

        if activations.ndim != 4:

            raise RuntimeError(
                "Grad-CAM target layer output must be 4D.\n"
                f"Got shape: {tuple(activations.shape)}"
            )

        if gradients.ndim != 4:

            raise RuntimeError(
                "Grad-CAM gradients must be 4D.\n"
                f"Got shape: {tuple(gradients.shape)}"
            )

        # Global average pooling of gradients.
        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        cam = (
            weights
            *
            activations
        ).sum(
            dim=1,
            keepdim=True
        )

        cam = torch.relu(
            cam
        )

        cam = torch.nn.functional.interpolate(
            cam,
            size=(
                224,
                224
            ),
            mode="bilinear",
            align_corners=False
        )

        cam = cam.squeeze(
            0
        ).squeeze(
            0
        )

        cam = cam.detach().cpu().numpy()

        cam_min = float(
            cam.min()
        )

        cam_max = float(
            cam.max()
        )

        if cam_max > cam_min:

            cam = (
                cam - cam_min
            ) / (
                cam_max - cam_min
            )

        else:

            cam = np.zeros_like(
                cam,
                dtype=np.float32
            )

        return (
            output.detach().cpu(),
            cam.astype(
                np.float32
            )
        )


gradcam = GradCAM(
    model,
    target_layer
)


# =============================================================================
# 16. PREDICTION CONSISTENCY + GRAD-CAM
# =============================================================================

print("\n" + "=" * 90)
print("RUNNING FROZEN CXR GRAD-CAM")
print("=" * 90)

results = []

prediction_mismatches = []

probability_differences = []

successful_maps = 0


for index, row in test_input_df.iterrows():

    condition_id = str(
        row["condition_id"]
    )

    target_binary = int(
        row["target_binary"]
    )

    npy_path = Path(
        row["npy_path"]
    )

    # -------------------------------------------------------------------------
    # Load frozen input.
    # -------------------------------------------------------------------------

    input_tensor = load_cxr_tensor(
        npy_path
    ).to(
        device
    )

    # -------------------------------------------------------------------------
    # Official frozen prediction.
    # -------------------------------------------------------------------------

    official_row = official_predictions[
        official_predictions[
            prediction_condition_col
        ]
        ==
        condition_id
    ]

    if len(official_row) != 1:

        raise RuntimeError(
            f"Official prediction row problem for "
            f"{condition_id}"
        )

    official_row = official_row.iloc[0]

    official_probability = float(
        official_row[
            "probability_DR"
        ]
    )

    official_prediction = int(
        official_row[
            "predicted_binary"
        ]
    )

    # -------------------------------------------------------------------------
    # Frozen model prediction + Grad-CAM.
    # -------------------------------------------------------------------------

    output, cam = gradcam.generate(
        input_tensor,
        target_class=official_prediction
    )

    logits = output[
        0
    ]

    probabilities_tensor = torch.softmax(
        logits,
        dim=0
    )

    probability_dr = float(
        probabilities_tensor[
            1
        ].item()
    )

    predicted_class = int(
        torch.argmax(
            probabilities_tensor
        ).item()
    )

    # -------------------------------------------------------------------------
    # Prediction consistency.
    # -------------------------------------------------------------------------

    probability_difference = abs(
        probability_dr
        -
        official_probability
    )

    probability_differences.append(
        probability_difference
    )

    if predicted_class != official_prediction:

        prediction_mismatches.append({

            "condition_id":
                condition_id,

            "official_prediction":
                official_prediction,

            "recomputed_prediction":
                predicted_class,

        })

    # -------------------------------------------------------------------------
    # Save Grad-CAM NPY.
    # -------------------------------------------------------------------------

    safe_id = re.sub(
        r"[^A-Za-z0-9_.-]",
        "_",
        condition_id
    )

    cam_npy_path = (
        CAM_NPY_ROOT
        / f"{safe_id}_GradCAM.npy"
    )

    np.save(
        cam_npy_path,
        cam
    )

    # -------------------------------------------------------------------------
    # Save overlay PNG.
    # -------------------------------------------------------------------------

    display_image = tensor_to_display_image(
        input_tensor
    )

    fig, ax = plt.subplots(
        figsize=(6, 6)
    )

    ax.imshow(
        display_image,
        cmap="gray"
    )

    ax.imshow(
        cam,
        cmap="jet",
        alpha=0.45,
        vmin=0,
        vmax=1
    )

    ax.axis(
        "off"
    )

    class_name = (
        "DR"
        if official_prediction == 1
        else "DS"
    )

    true_name = (
        "DR"
        if target_binary == 1
        else "DS"
    )

    ax.set_title(
        f"Condition: {condition_id}\n"
        f"True: {true_name} | "
        f"Predicted: {class_name} | "
        f"P(DR): {probability_dr:.4f}",
        fontsize=10
    )

    fig.tight_layout(
        pad=0.5
    )

    cam_png_path = (
        CAM_PNG_ROOT
        / f"{safe_id}_GradCAM.png"
    )

    fig.savefig(
        cam_png_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(
        fig
    )

    # -------------------------------------------------------------------------
    # CAM statistics.
    # -------------------------------------------------------------------------

    high_activation_fraction = float(
        (
            cam >= 0.5
        ).mean()
    )

    mean_activation = float(
        cam.mean()
    )

    max_activation = float(
        cam.max()
    )

    results.append({

        "condition_id":
            condition_id,

        "target_binary":
            target_binary,

        "true_class":
            (
                "DR"
                if target_binary == 1
                else "DS"
            ),

        "official_predicted_binary":
            official_prediction,

        "recomputed_predicted_binary":
            predicted_class,

        "official_probability_DR":
            official_probability,

        "recomputed_probability_DR":
            probability_dr,

        "probability_absolute_difference":
            probability_difference,

        "gradcam_mean":
            mean_activation,

        "gradcam_max":
            max_activation,

        "gradcam_fraction_ge_0_5":
            high_activation_fraction,

        "npy_path":
            str(npy_path),

        "gradcam_npy_path":
            str(cam_npy_path),

        "gradcam_png_path":
            str(cam_png_path),

    })

    successful_maps += 1

    if (
        (index + 1) % 25 == 0
        or
        (index + 1) == 396
    ):

        print(
            f"Processed "
            f"{index + 1:>3}/396"
        )


# =============================================================================
# 17. REMOVE GRAD-CAM HOOKS
# =============================================================================

gradcam.remove_hooks()


# =============================================================================
# 18. PREDICTION CONSISTENCY GATE
# =============================================================================

print("\n" + "=" * 90)
print("FROZEN PREDICTION CONSISTENCY")
print("=" * 90)

if successful_maps != 396:

    raise RuntimeError(
        f"Expected 396 Grad-CAM maps, "
        f"generated {successful_maps}."
    )

if len(prediction_mismatches) != 0:

    raise RuntimeError(
        "Recomputed CXR predictions do not match "
        "the official frozen predictions.\n"
        f"Mismatches: {prediction_mismatches[:10]}"
    )

max_probability_difference = max(
    probability_differences
)

if max_probability_difference > 1e-5:

    raise RuntimeError(
        "Recomputed probabilities differ from official "
        f"frozen probabilities.\n"
        f"Maximum absolute difference: "
        f"{max_probability_difference:.10f}"
    )

print(
    "Prediction class agreement : 396/396"
)

print(
    "Prediction agreement       : 100.00%"
)

print(
    "Maximum probability diff   : "
    f"{max_probability_difference:.10f}"
)

print(
    "Frozen prediction audit    : PASS"
)


# =============================================================================
# 19. GRAD-CAM RESULT TABLE
# =============================================================================

gradcam_df = pd.DataFrame(
    results
)

if len(gradcam_df) != 396:

    raise RuntimeError(
        "Grad-CAM result table does not contain 396 rows."
    )

if gradcam_df[
    "condition_id"
].nunique() != 396:

    raise RuntimeError(
        "Grad-CAM result table contains duplicate conditions."
    )


GRADCAM_RESULTS_FILE = (
    CELL2_ROOT
    / "Notebook17_Cell2_FINAL_396_CXR_GradCAM_Results.csv"
)

gradcam_df.to_csv(
    GRADCAM_RESULTS_FILE,
    index=False
)


# =============================================================================
# 20. TEST-CLASS SUMMARY
# =============================================================================

summary_by_class = (
    gradcam_df
    .groupby(
        "true_class"
    )
    .agg(
        n=(
            "condition_id",
            "count"
        ),

        mean_gradcam_activation=(
            "gradcam_mean",
            "mean"
        ),

        mean_max_gradcam=(
            "gradcam_max",
            "mean"
        ),

        mean_high_activation_fraction=(
            "gradcam_fraction_ge_0_5",
            "mean"
        ),

    )
    .reset_index()
)

SUMMARY_CLASS_FILE = (
    CELL2_ROOT
    / "Notebook17_Cell2_CXR_GradCAM_Class_Summary.csv"
)

summary_by_class.to_csv(
    SUMMARY_CLASS_FILE,
    index=False
)


# =============================================================================
# 21. OVERALL GRAD-CAM SUMMARY
# =============================================================================

overall_summary = {

    "total_test_conditions":
        396,

    "gradcam_maps_generated":
        successful_maps,

    "prediction_class_agreement":
        1.0,

    "maximum_probability_absolute_difference":
        float(
            max_probability_difference
        ),

    "target_layer":
        target_layer_name,

    "input_shape":
        [
            1,
            3,
            224,
            224
        ],

    "stored_input_range":
        "[-1,1]",

    "explainability_preprocessing":
        "[-1,1] -> [0,1] -> ImageNet normalization",

    "augmentation_used":
        False,

    "training_performed":
        False,

    "fine_tuning_performed":
        False,

    "checkpoint_selection_performed":
        False,

    "threshold_tuning_performed":
        False,

    "test_predictions_modified":
        False,

    "outputs": {

        "gradcam_results":
            str(GRADCAM_RESULTS_FILE),

        "class_summary":
            str(SUMMARY_CLASS_FILE),

        "npy_directory":
            str(CAM_NPY_ROOT),

        "png_directory":
            str(CAM_PNG_ROOT),

    },

}


SUMMARY_FILE = (
    CELL2_ROOT
    / "Notebook17_Cell2_CXR_GradCAM_Summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        overall_summary,
        f,
        indent=4
    )


# =============================================================================
# 22. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 90)
print("NOTEBOOK 17 — CELL 2 STATUS: PASS")
print("=" * 90)

print(
    "Frozen CXR test conditions : 396"
)

print(
    "Grad-CAM maps generated    :",
    successful_maps
)

print(
    "Prediction agreement       : 396/396"
)

print(
    "Maximum probability diff   : "
    f"{max_probability_difference:.10f}"
)

print(
    "Target layer               :",
    target_layer_name
)

print(
    "\nTraining performed         : NO"
)

print(
    "Fine-tuning performed     : NO"
)

print(
    "Checkpoint selection      : NO"
)

print(
    "Threshold tuning          : NO"
)

print(
    "Test predictions modified : NO"
)

print(
    "\nGrad-CAM CSV:"
)

print(
    GRADCAM_RESULTS_FILE
)

print(
    "\nGrad-CAM NPY directory:"
)

print(
    CAM_NPY_ROOT
)

print(
    "\nGrad-CAM PNG directory:"
)

print(
    CAM_PNG_ROOT
)

print(
    "\nSummary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nCELL 2 COMPLETE."
)

print(
    "Proceed to Notebook 17 — Cell 3."
)

NOTEBOOK 17 — CELL 2
FROZEN CoAtNet CXR Grad-CAM EXPLAINABILITY
CXR checkpoint              : PASS
CXR frozen predictions      : PASS
Authoritative split         : PASS
CXR input root              : PASS
CXR NPY root                : PASS
CXR manifest                : PASS
Cell 1 audit                : PASS

Cell 1 safety gate: PASS
Authoritative test conditions: 396
Official CXR predictions: 396/396 aligned

Manifest rows: 1976

Resolving frozen test NPY inputs...
Resolved test NPY inputs: 396
All 396 NPY files: PASS

LOADING FROZEN CoAtNet RUN-2
Checkpoint loaded: PASS

Grad-CAM target layer:
stages.3.blocks.1.mlp.fc2
Conv2d(3072, 768, kernel_size=(1, 1), stride=(1, 1))

RUNNING FROZEN CXR GRAD-CAM
Processed  25/396
Processed  50/396
Processed  75/396
Processed 100/396
Processed 125/396
Processed 150/396
Processed 175/396
Processed 200/396
Processed 225/396
Processed 250/396
Processed 275/396
Processed 300/396
Processed 325/396
Processed 350/396
Processed 375/396
Processed 396/396

F

In [12]:
# =============================================================================
# NOTEBOOK 17 — CELL 3
# FINAL PUBLICATION FIGURES
# WINDOWS PATH-LENGTH SAFE VERSION
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 1. USE A SHORT OUTPUT PATH
# =============================================================================

FINAL_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT"
)

# IMPORTANT:
# Short directory specifically avoids Windows MAX_PATH problems.
FIGURE_ROOT = FINAL_ROOT / "Notebook17_Figures"

FIGURE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("NOTEBOOK 17 — CELL 3")
print("FINAL PUBLICATION FIGURES")
print("=" * 90)

print("Figure directory:")
print(FIGURE_ROOT)

print("Directory exists:", FIGURE_ROOT.exists())


# =============================================================================
# 2. VERIFY ANALYSIS DATA
# =============================================================================

if "analysis_df" not in globals():
    raise RuntimeError(
        "analysis_df is not available in the current notebook session. "
        "Do not rerun Grad-CAM. Run the successful Cell 3 analysis cell first."
    )

if len(analysis_df) != 396:
    raise RuntimeError(
        f"Expected 396 Grad-CAM records, found {len(analysis_df)}."
    )

required_columns = [
    "true_class",
    "prediction_status",
    "map_mean",
    "fraction_ge_0_25",
    "fraction_ge_0_50",
    "fraction_ge_0_75",
    "fraction_ge_0_90",
]

missing = [
    c for c in required_columns
    if c not in analysis_df.columns
]

if missing:
    raise RuntimeError(
        "Missing analysis columns:\n" + "\n".join(missing)
    )


# =============================================================================
# 3. FIGURE 1 — DS VS DR ACTIVATION
# =============================================================================

FIG1 = FIGURE_ROOT / "Fig1_DS_DR_GradCAM.png"

ds_values = analysis_df.loc[
    analysis_df["true_class"] == "DS",
    "map_mean"
].astype(float).to_numpy()

dr_values = analysis_df.loc[
    analysis_df["true_class"] == "DR",
    "map_mean"
].astype(float).to_numpy()

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.boxplot(
    [
        ds_values,
        dr_values
    ],
    tick_labels=[
        "DS-TB",
        "DR-TB"
    ],
    showfliers=False
)

ax.set_ylabel(
    "Mean Grad-CAM activation"
)

ax.set_title(
    "Grad-CAM Activation by True Class"
)

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    FIG1,
    dpi=600,
    format="png",
    bbox_inches="tight"
)

plt.close(fig)

print("Figure 1: PASS")


# =============================================================================
# 4. FIGURE 2 — ACTIVATION CONCENTRATION
# =============================================================================

FIG2 = FIGURE_ROOT / "Fig2_GradCAM_Concentration.png"

thresholds = np.array(
    [
        0.25,
        0.50,
        0.75,
        0.90
    ],
    dtype=float
)

means = np.array(
    [
        analysis_df[
            "fraction_ge_0_25"
        ].astype(float).mean(),

        analysis_df[
            "fraction_ge_0_50"
        ].astype(float).mean(),

        analysis_df[
            "fraction_ge_0_75"
        ].astype(float).mean(),

        analysis_df[
            "fraction_ge_0_90"
        ].astype(float).mean(),
    ],
    dtype=float
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    thresholds,
    means,
    marker="o"
)

ax.set_xlabel(
    "Grad-CAM activation threshold"
)

ax.set_ylabel(
    "Mean fraction of image"
)

ax.set_title(
    "Grad-CAM Activation Concentration"
)

ax.set_xticks(
    thresholds
)

ax.grid(
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    FIG2,
    dpi=600,
    format="png",
    bbox_inches="tight"
)

plt.close(fig)

print("Figure 2: PASS")


# =============================================================================
# 5. FIGURE 3 — CORRECT VS INCORRECT
# =============================================================================

FIG3 = FIGURE_ROOT / "Fig3_Correct_Incorrect_GradCAM.png"

correct_values = analysis_df.loc[
    analysis_df["prediction_status"] == "Correct",
    "map_mean"
].astype(float).to_numpy()

incorrect_values = analysis_df.loc[
    analysis_df["prediction_status"] == "Incorrect",
    "map_mean"
].astype(float).to_numpy()

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.boxplot(
    [
        correct_values,
        incorrect_values
    ],
    tick_labels=[
        "Correct",
        "Incorrect"
    ],
    showfliers=False
)

ax.set_ylabel(
    "Mean Grad-CAM activation"
)

ax.set_title(
    "Grad-CAM Activation by Prediction Correctness"
)

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    FIG3,
    dpi=600,
    format="png",
    bbox_inches="tight"
)

plt.close(fig)

print("Figure 3: PASS")


# =============================================================================
# 6. FIGURE 4 — DS VS DR ACTIVATION AREA
# =============================================================================

FIG4 = FIGURE_ROOT / "Fig4_DS_DR_Activation_Area.png"

ds_data = analysis_df.loc[
    analysis_df["true_class"] == "DS"
]

dr_data = analysis_df.loc[
    analysis_df["true_class"] == "DR"
]

positions = np.array(
    [
        1,
        2,
        3,
        4
    ]
)

ds_means = np.array(
    [
        ds_data["fraction_ge_0_25"].astype(float).mean(),
        ds_data["fraction_ge_0_50"].astype(float).mean(),
        ds_data["fraction_ge_0_75"].astype(float).mean(),
        ds_data["fraction_ge_0_90"].astype(float).mean(),
    ]
)

dr_means = np.array(
    [
        dr_data["fraction_ge_0_25"].astype(float).mean(),
        dr_data["fraction_ge_0_50"].astype(float).mean(),
        dr_data["fraction_ge_0_75"].astype(float).mean(),
        dr_data["fraction_ge_0_90"].astype(float).mean(),
    ]
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    positions,
    ds_means,
    marker="o",
    label="DS-TB"
)

ax.plot(
    positions,
    dr_means,
    marker="o",
    label="DR-TB"
)

ax.set_xticks(
    positions
)

ax.set_xticklabels(
    [
        "≥0.25",
        "≥0.50",
        "≥0.75",
        "≥0.90"
    ]
)

ax.set_xlabel(
    "Grad-CAM activation threshold"
)

ax.set_ylabel(
    "Mean fraction of image"
)

ax.set_title(
    "Grad-CAM Activation Area by True Class"
)

ax.legend()

ax.grid(
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    FIG4,
    dpi=600,
    format="png",
    bbox_inches="tight"
)

plt.close(fig)

print("Figure 4: PASS")


# =============================================================================
# 7. VERIFY ALL FIGURES
# =============================================================================

figure_files = [
    FIG1,
    FIG2,
    FIG3,
    FIG4
]

for path in figure_files:

    if not path.exists():
        raise RuntimeError(
            f"Figure was not created:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Figure is empty:\n{path}"
        )


# =============================================================================
# 8. MANIFEST
# =============================================================================

MANIFEST = FIGURE_ROOT / "Figure_Manifest.csv"

manifest_df = pd.DataFrame({

    "figure": [
        "Figure 1",
        "Figure 2",
        "Figure 3",
        "Figure 4"
    ],

    "description": [
        "Grad-CAM activation by true class",
        "Grad-CAM activation concentration",
        "Grad-CAM activation by prediction correctness",
        "Grad-CAM activation area by true class"
    ],

    "file": [
        str(FIG1),
        str(FIG2),
        str(FIG3),
        str(FIG4)
    ],

    "dpi": [
        600,
        600,
        600,
        600
    ]
})

manifest_df.to_csv(
    MANIFEST,
    index=False
)


# =============================================================================
# 9. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 90)
print("NOTEBOOK 17 — CELL 3 FIGURES STATUS: PASS")
print("=" * 90)

print("Figures generated : 4")
print("Resolution         : 600 DPI")

print("\nSaved to:")
print(FIGURE_ROOT)

for path in figure_files:
    print("\n", path)

print("\nManifest:")
print(MANIFEST)

print("\nALL FOUR FIGURES VERIFIED SUCCESSFULLY.")

NOTEBOOK 17 — CELL 3
FINAL PUBLICATION FIGURES
Figure directory:
C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Figures
Directory exists: True
Figure 1: PASS
Figure 2: PASS
Figure 3: PASS
Figure 4: PASS


NOTEBOOK 17 — CELL 3 FIGURES STATUS: PASS
Figures generated : 4
Resolution         : 600 DPI

Saved to:
C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Figures

 C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Figures\Fig1_DS_DR_GradCAM.png

 C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Figures\Fig2_GradCAM_Concentration.png

 C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Figures\Fig3_Correct_Incorrect_GradCAM.png

 C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_P

In [13]:
# =============================================================================
# NOTEBOOK 17 — CELL 4
# STATISTICAL ANALYSIS OF FROZEN CXR GRAD-CAM RESULTS
# =============================================================================
#
# IMPORTANT
# ----------
# Analysis only.
#
# NO:
#   - training
#   - fine-tuning
#   - model inference
#   - checkpoint selection
#   - threshold tuning
#   - modification of predictions
#
# This cell statistically analyzes the Grad-CAM measurements already generated
# in Cells 2 and 3.
#
# Analyses:
#   1. DS vs DR activation
#   2. Correct vs incorrect activation
#   3. Mann-Whitney U tests
#   4. Bootstrap 95% confidence intervals
#   5. Rank-biserial effect size
#   6. Cohen's d
#   7. Multiple-comparison correction
#   8. Publication-ready statistical table
# =============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from scipy.stats import (
    mannwhitneyu,
    rankdata,
    norm
)


# =============================================================================
# 1. PATHS
# =============================================================================

FINAL_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT"
)

CELL4_ROOT = (
    FINAL_ROOT
    / "Notebook17_Cell4_GradCAM_Statistical_Analysis"
)

CELL4_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("NOTEBOOK 17 — CELL 4")
print("STATISTICAL ANALYSIS OF FROZEN CXR GRAD-CAM RESULTS")
print("=" * 90)

print("Output directory:")
print(CELL4_ROOT)


# =============================================================================
# 2. VERIFY CELL 3 DATA
# =============================================================================

if "analysis_df" not in globals():
    raise RuntimeError(
        "analysis_df is not available. "
        "Run the successful Notebook 17 Cell 3 analysis first."
    )

if len(analysis_df) != 396:
    raise RuntimeError(
        f"Expected 396 test cases, found {len(analysis_df)}."
    )

required_columns = [
    "condition_id",
    "target_binary",
    "true_class",
    "prediction_status",
    "map_mean",
    "map_max",
    "fraction_ge_0_25",
    "fraction_ge_0_50",
    "fraction_ge_0_75",
    "fraction_ge_0_90",
    "confidence",
]

missing_columns = [
    col
    for col in required_columns
    if col not in analysis_df.columns
]

if missing_columns:
    raise RuntimeError(
        "Required Cell 3 columns are missing:\n"
        + "\n".join(missing_columns)
    )

print("\nCell 3 analysis data       : PASS")
print("Test cases                :", len(analysis_df))


# =============================================================================
# 3. SAFETY AUDIT
# =============================================================================

if analysis_df["condition_id"].nunique() != 396:
    raise RuntimeError(
        "Condition IDs are not unique."
    )

if not analysis_df["target_binary"].isin([0, 1]).all():
    raise RuntimeError(
        "Invalid target values detected."
    )

if not np.isfinite(
    analysis_df[
        [
            "map_mean",
            "map_max",
            "fraction_ge_0_25",
            "fraction_ge_0_50",
            "fraction_ge_0_75",
            "fraction_ge_0_90",
            "confidence",
        ]
    ].to_numpy()
).all():
    raise RuntimeError(
        "Non-finite Grad-CAM/statistical values detected."
    )

print("Condition uniqueness       : PASS")
print("Target audit               : PASS")
print("Finite-value audit         : PASS")


# =============================================================================
# 4. STATISTICAL HELPER FUNCTIONS
# =============================================================================

def cohens_d(x, y):
    """
    Standardized mean difference using pooled sample standard deviation.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    nx = len(x)
    ny = len(y)

    if nx < 2 or ny < 2:
        return np.nan

    sx = np.std(x, ddof=1)
    sy = np.std(y, ddof=1)

    pooled_sd = np.sqrt(
        (
            (nx - 1) * sx**2
            +
            (ny - 1) * sy**2
        )
        /
        (
            nx + ny - 2
        )
    )

    if pooled_sd == 0:
        return 0.0

    return (
        (np.mean(x) - np.mean(y))
        /
        pooled_sd
    )


def rank_biserial_from_u(
    u_value,
    nx,
    ny
):
    """
    Rank-biserial effect size derived from the Mann-Whitney U statistic.

    Positive value means the first group tends to have larger values.
    """
    if nx == 0 or ny == 0:
        return np.nan

    return (
        (2.0 * u_value)
        /
        (nx * ny)
    ) - 1.0


def bootstrap_mean_difference(
    x,
    y,
    n_boot=2000,
    seed=42
):
    """
    Bootstrap CI for mean(x) - mean(y).

    Bootstrap is descriptive uncertainty estimation only.
    It is NOT used for model selection or threshold selection.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    rng = np.random.default_rng(
        seed
    )

    x_indices = rng.integers(
        0,
        len(x),
        size=(
            n_boot,
            len(x)
        )
    )

    y_indices = rng.integers(
        0,
        len(y),
        size=(
            n_boot,
            len(y)
        )
    )

    x_samples = x[
        x_indices
    ]

    y_samples = y[
        y_indices
    ]

    differences = (
        x_samples.mean(axis=1)
        -
        y_samples.mean(axis=1)
    )

    ci_low, ci_high = np.percentile(
        differences,
        [2.5, 97.5]
    )

    return (
        float(
            differences.mean()
        ),
        float(ci_low),
        float(ci_high)
    )


def bootstrap_median_difference(
    x,
    y,
    n_boot=2000,
    seed=42
):
    """
    Bootstrap CI for median(x) - median(y).
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    rng = np.random.default_rng(
        seed
    )

    x_indices = rng.integers(
        0,
        len(x),
        size=(
            n_boot,
            len(x)
        )
    )

    y_indices = rng.integers(
        0,
        len(y),
        size=(
            n_boot,
            len(y)
        )
    )

    x_samples = x[
        x_indices
    ]

    y_samples = y[
        y_indices
    ]

    differences = (
        np.median(
            x_samples,
            axis=1
        )
        -
        np.median(
            y_samples,
            axis=1
        )
    )

    ci_low, ci_high = np.percentile(
        differences,
        [2.5, 97.5]
    )

    return (
        float(
            differences.mean()
        ),
        float(ci_low),
        float(ci_high)
    )


# =============================================================================
# 5. STATISTICAL TEST FUNCTION
# =============================================================================

def compare_groups(
    x,
    y,
    group1_name,
    group2_name,
    variable_name,
    bootstrap_seed
):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    if len(x) == 0 or len(y) == 0:
        raise RuntimeError(
            f"Empty group encountered for {variable_name}."
        )

    # Mann-Whitney U.
    # Two-sided because there is no directional threshold/model decision here.
    u_stat, p_value = mannwhitneyu(
        x,
        y,
        alternative="two-sided",
        method="auto"
    )

    mean_difference = (
        float(np.mean(x))
        -
        float(np.mean(y))
    )

    median_difference = (
        float(np.median(x))
        -
        float(np.median(y))
    )

    mean_bootstrap, mean_ci_low, mean_ci_high = (
        bootstrap_mean_difference(
            x,
            y,
            n_boot=2000,
            seed=bootstrap_seed
        )
    )

    median_bootstrap, median_ci_low, median_ci_high = (
        bootstrap_median_difference(
            x,
            y,
            n_boot=2000,
            seed=bootstrap_seed + 100
        )
    )

    return {

        "group_1":
            group1_name,

        "group_2":
            group2_name,

        "variable":
            variable_name,

        "n_group_1":
            int(len(x)),

        "n_group_2":
            int(len(y)),

        "mean_group_1":
            float(np.mean(x)),

        "mean_group_2":
            float(np.mean(y)),

        "median_group_1":
            float(np.median(x)),

        "median_group_2":
            float(np.median(y)),

        "mean_difference_group1_minus_group2":
            mean_difference,

        "median_difference_group1_minus_group2":
            median_difference,

        "mean_difference_bootstrap":
            mean_bootstrap,

        "mean_difference_ci95_low":
            mean_ci_low,

        "mean_difference_ci95_high":
            mean_ci_high,

        "median_difference_bootstrap":
            median_bootstrap,

        "median_difference_ci95_low":
            median_ci_low,

        "median_difference_ci95_high":
            median_ci_high,

        "mann_whitney_U":
            float(u_stat),

        "mann_whitney_p":
            float(p_value),

        "cohens_d":
            float(
                cohens_d(
                    x,
                    y
                )
            ),

        "rank_biserial":
            float(
                rank_biserial_from_u(
                    u_stat,
                    len(x),
                    len(y)
                )
            ),

    }


# =============================================================================
# 6. DEFINE GRAD-CAM VARIABLES
# =============================================================================

variables = {

    "Mean activation":
        "map_mean",

    "Maximum activation":
        "map_max",

    "Area ≥ 0.25":
        "fraction_ge_0_25",

    "Area ≥ 0.50":
        "fraction_ge_0_50",

    "Area ≥ 0.75":
        "fraction_ge_0_75",

    "Area ≥ 0.90":
        "fraction_ge_0_90",

}


# =============================================================================
# 7. DS VS DR STATISTICAL COMPARISON
# =============================================================================

print("\n" + "=" * 90)
print("DS VS DR STATISTICAL ANALYSIS")
print("=" * 90)

ds_mask = (
    analysis_df["target_binary"] == 0
)

dr_mask = (
    analysis_df["target_binary"] == 1
)

ds_df = analysis_df.loc[
    ds_mask
]

dr_df = analysis_df.loc[
    dr_mask
]

print(
    "DS n:",
    len(ds_df)
)

print(
    "DR n:",
    len(dr_df)
)


ds_dr_results = []

for index, (
    variable_label,
    column_name
) in enumerate(
    variables.items()
):

    result = compare_groups(

        ds_df[column_name].values,

        dr_df[column_name].values,

        "DS-TB",

        "DR-TB",

        variable_label,

        bootstrap_seed=4200 + index

    )

    ds_dr_results.append(
        result
    )


ds_dr_results_df = pd.DataFrame(
    ds_dr_results
)


# =============================================================================
# 8. MULTIPLE-COMPARISON CORRECTION
# =============================================================================
#
# Six related Grad-CAM variables are tested for DS vs DR.
# Benjamini-Hochberg FDR correction is applied.
# =============================================================================

def benjamini_hochberg(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    m = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted = np.empty(
        m,
        dtype=float
    )

    running = 1.0

    for i in range(
        m - 1,
        -1,
        -1
    ):

        rank = i + 1

        value = (
            ranked[i]
            *
            m
            /
            rank
        )

        running = min(
            running,
            value
        )

        adjusted[i] = running

    result = np.empty(
        m,
        dtype=float
    )

    result[
        order
    ] = adjusted

    return np.clip(
        result,
        0.0,
        1.0
    )


ds_dr_results_df[
    "mann_whitney_p_fdr"
] = benjamini_hochberg(
    ds_dr_results_df[
        "mann_whitney_p"
    ].values
)

ds_dr_results_df[
    "fdr_significant_alpha_0_05"
] = (
    ds_dr_results_df[
        "mann_whitney_p_fdr"
    ]
    <
    0.05
)


# =============================================================================
# 9. PRINT DS VS DR RESULTS
# =============================================================================

display_columns = [

    "variable",

    "n_group_1",
    "n_group_2",

    "mean_group_1",
    "mean_group_2",

    "median_group_1",
    "median_group_2",

    "mean_difference_group1_minus_group2",

    "mean_difference_ci95_low",
    "mean_difference_ci95_high",

    "mann_whitney_U",
    "mann_whitney_p",

    "mann_whitney_p_fdr",

    "cohens_d",
    "rank_biserial",

    "fdr_significant_alpha_0_05",

]

print(
    ds_dr_results_df[
        display_columns
    ].to_string(
        index=False
    )
)


# =============================================================================
# 10. CORRECT VS INCORRECT ANALYSIS
# =============================================================================

print("\n" + "=" * 90)
print("CORRECT VS INCORRECT STATISTICAL ANALYSIS")
print("=" * 90)

correct_df = analysis_df.loc[
    analysis_df["prediction_status"] == "Correct"
]

incorrect_df = analysis_df.loc[
    analysis_df["prediction_status"] == "Incorrect"
]

print(
    "Correct n:",
    len(correct_df)
)

print(
    "Incorrect n:",
    len(incorrect_df)
)


correctness_results = []

for index, (
    variable_label,
    column_name
) in enumerate(
    variables.items()
):

    result = compare_groups(

        correct_df[column_name].values,

        incorrect_df[column_name].values,

        "Correct",

        "Incorrect",

        variable_label,

        bootstrap_seed=8400 + index

    )

    correctness_results.append(
        result
    )


correctness_results_df = pd.DataFrame(
    correctness_results
)

correctness_results_df[
    "mann_whitney_p_fdr"
] = benjamini_hochberg(
    correctness_results_df[
        "mann_whitney_p"
    ].values
)

correctness_results_df[
    "fdr_significant_alpha_0_05"
] = (
    correctness_results_df[
        "mann_whitney_p_fdr"
    ]
    <
    0.05
)


print(
    correctness_results_df[
        display_columns
    ].to_string(
        index=False
    )
)


# =============================================================================
# 11. CONFIDENCE VS GRAD-CAM ACTIVATION
# =============================================================================

print("\n" + "=" * 90)
print("CONFIDENCE VS GRAD-CAM ASSOCIATION")
print("=" * 90)

confidence_activation = analysis_df[
    [
        "confidence",
        "map_mean",
        "fraction_ge_0_50",
        "fraction_ge_0_75",
        "fraction_ge_0_90",
    ]
].corr(
    method="spearman"
)

print(
    confidence_activation.to_string()
)


# =============================================================================
# 12. SAVE STATISTICAL RESULTS
# =============================================================================

DS_DR_FILE = (
    CELL4_ROOT
    / "Notebook17_Cell4_DS_vs_DR_GradCAM_Statistics.csv"
)

CORRECTNESS_FILE = (
    CELL4_ROOT
    / "Notebook17_Cell4_Correct_vs_Incorrect_GradCAM_Statistics.csv"
)

CORRELATION_FILE = (
    CELL4_ROOT
    / "Notebook17_Cell4_Confidence_GradCAM_Spearman_Correlation.csv"
)

ds_dr_results_df.to_csv(
    DS_DR_FILE,
    index=False
)

correctness_results_df.to_csv(
    CORRECTNESS_FILE,
    index=False
)

confidence_activation.to_csv(
    CORRELATION_FILE
)


# =============================================================================
# 13. COMBINED PUBLICATION TABLE
# =============================================================================

publication_table = pd.concat(
    [
        ds_dr_results_df.assign(
            comparison="DS-TB vs DR-TB"
        ),
        correctness_results_df.assign(
            comparison="Correct vs Incorrect"
        ),
    ],
    ignore_index=True
)

PUBLICATION_TABLE = (
    CELL4_ROOT
    / "Notebook17_Cell4_FINAL_GradCAM_Statistical_Analysis_Table.csv"
)

publication_table.to_csv(
    PUBLICATION_TABLE,
    index=False
)


# =============================================================================
# 14. SAVE SUMMARY JSON
# =============================================================================

SUMMARY_FILE = (
    CELL4_ROOT
    / "Notebook17_Cell4_GradCAM_Statistical_Analysis_Summary.json"
)

summary = {

    "status":
        "PASS",

    "analysis_type":
        "Frozen Grad-CAM statistical analysis",

    "test_cases":
        396,

    "DS_cases":
        int(len(ds_df)),

    "DR_cases":
        int(len(dr_df)),

    "correct_cases":
        int(len(correct_df)),

    "incorrect_cases":
        int(len(incorrect_df)),

    "statistical_test":
        "Two-sided Mann-Whitney U",

    "effect_sizes": [
        "Cohen's d",
        "Rank-biserial correlation"
    ],

    "confidence_intervals":
        "2000-sample nonparametric bootstrap 95% CI",

    "multiple_comparison_correction":
        "Benjamini-Hochberg FDR",

    "fdr_alpha":
        0.05,

    "training_performed":
        False,

    "fine_tuning_performed":
        False,

    "model_inference_performed":
        False,

    "checkpoint_selection_performed":
        False,

    "threshold_tuning_performed":
        False,

    "test_predictions_modified":
        False,

    "outputs": {

        "DS_vs_DR":
            str(DS_DR_FILE),

        "Correct_vs_Incorrect":
            str(CORRECTNESS_FILE),

        "Confidence_correlation":
            str(CORRELATION_FILE),

        "Publication_table":
            str(PUBLICATION_TABLE),

        "Summary":
            str(SUMMARY_FILE),

    },

}

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 15. FINAL OUTPUT EXISTENCE AUDIT
# =============================================================================

required_outputs = [

    DS_DR_FILE,
    CORRECTNESS_FILE,
    CORRELATION_FILE,
    PUBLICATION_TABLE,
    SUMMARY_FILE,

]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:

    raise RuntimeError(
        "Cell 4 output files missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# =============================================================================
# 16. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 90)
print("NOTEBOOK 17 — CELL 4 STATUS: PASS")
print("=" * 90)

print(
    "Test cases analyzed       :",
    len(analysis_df)
)

print(
    "DS cases                  :",
    len(ds_df)
)

print(
    "DR cases                  :",
    len(dr_df)
)

print(
    "Correct cases             :",
    len(correct_df)
)

print(
    "Incorrect cases           :",
    len(incorrect_df)
)

print(
    "\nStatistical test          : Two-sided Mann-Whitney U"
)

print(
    "Effect sizes              : Cohen's d + rank-biserial"
)

print(
    "Bootstrap                 : 2000 iterations"
)

print(
    "Multiple comparison       : Benjamini-Hochberg FDR"
)

print(
    "\nTraining performed        : NO"
)

print(
    "Fine-tuning performed     : NO"
)

print(
    "Model inference           : NO"
)

print(
    "Checkpoint selection      : NO"
)

print(
    "Threshold tuning          : NO"
)

print(
    "Test predictions modified : NO"
)

print("\nOutput directory:")
print(CELL4_ROOT)

print("\nPublication statistical table:")
print(PUBLICATION_TABLE)

print("\nSummary:")
print(SUMMARY_FILE)

print("\nCELL 4 COMPLETE.")
print("Proceed to Notebook 17 — Cell 5.")

NOTEBOOK 17 — CELL 4
STATISTICAL ANALYSIS OF FROZEN CXR GRAD-CAM RESULTS
Output directory:
C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Cell4_GradCAM_Statistical_Analysis

Cell 3 analysis data       : PASS
Test cases                : 396
Condition uniqueness       : PASS
Target audit               : PASS
Finite-value audit         : PASS

DS VS DR STATISTICAL ANALYSIS
DS n: 141
DR n: 255
          variable  n_group_1  n_group_2  mean_group_1  mean_group_2  median_group_1  median_group_2  mean_difference_group1_minus_group2  mean_difference_ci95_low  mean_difference_ci95_high  mann_whitney_U  mann_whitney_p  mann_whitney_p_fdr  cohens_d  rank_biserial  fdr_significant_alpha_0_05
   Mean activation        141        255      0.146094      0.146241        0.141560        0.141966                        -1.472981e-04                 -0.009151                   0.009315         17816.0        0.882644                 1.0 -

In [15]:
# =============================================================================
# NOTEBOOK 17 — CELL 5
# REPRESENTATIVE CXR GRAD-CAM VISUALIZATION
# CORRECTED — NO REPRESENTATIVE CSV REQUIRED
# =============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


# =============================================================================
# 1. FROZEN PATHS
# =============================================================================

FINAL_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT"
)

NOTEBOOK17_ROOT = (
    FINAL_ROOT
    / "Notebook17_Final_Explainability_Analysis"
)

CELL2_ROOT = (
    NOTEBOOK17_ROOT
    / "Cell2_CXR_GradCAM"
)

NPY_ROOT = (
    CELL2_ROOT
    / "GradCAM_Maps_NPY"
)

PNG_ROOT = (
    CELL2_ROOT
    / "GradCAM_Overlays_PNG"
)

OUTPUT_ROOT = (
    FINAL_ROOT
    / "Notebook17_Cell5_Representative_GradCAM"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("NOTEBOOK 17 — CELL 5")
print("REPRESENTATIVE CXR GRAD-CAM VISUALIZATION")
print("=" * 90)

print("Output directory:")
print(OUTPUT_ROOT)


# =============================================================================
# 2. VERIFY CELL 3 DATA
# =============================================================================

if "analysis_df" not in globals():
    raise RuntimeError(
        "analysis_df is not available in the current notebook session."
    )

if len(analysis_df) != 396:
    raise RuntimeError(
        f"Expected 396 Grad-CAM records, found {len(analysis_df)}."
    )

required_columns = [
    "condition_id",
    "true_class",
    "prediction_status",
    "map_mean",
]

missing = [
    c
    for c in required_columns
    if c not in analysis_df.columns
]

if missing:
    raise RuntimeError(
        "Required columns missing from analysis_df:\n"
        + "\n".join(missing)
    )

print("Cell 3 analysis data : PASS")
print("Records              :", len(analysis_df))


# =============================================================================
# 3. VERIFY CELL 2 DIRECTORIES
# =============================================================================

if not NPY_ROOT.exists():
    raise FileNotFoundError(
        f"Grad-CAM NPY directory not found:\n{NPY_ROOT}"
    )

if not PNG_ROOT.exists():
    raise FileNotFoundError(
        f"Grad-CAM PNG directory not found:\n{PNG_ROOT}"
    )

print("Grad-CAM NPY root    : PASS")
print("Grad-CAM PNG root    : PASS")


# =============================================================================
# 4. NORMALIZE CONDITION IDs
# =============================================================================

analysis_df = analysis_df.copy()

analysis_df["condition_id"] = (
    analysis_df["condition_id"]
    .astype(str)
    .str.strip()
)


# =============================================================================
# 5. SELECT FOUR REPRESENTATIVE CASES
# =============================================================================
#
# Selection rule:
#
# DS Correct:
#     highest confidence among correctly classified DS cases
#
# DR Correct:
#     highest confidence among correctly classified DR cases
#
# DS Incorrect:
#     highest confidence among incorrectly classified DS cases
#
# DR Incorrect:
#     highest confidence among incorrectly classified DR cases
#
# This is descriptive case selection only.
# It does NOT alter the model or predictions.
# =============================================================================

if "confidence" in analysis_df.columns:

    confidence_column = "confidence"

elif "probability_DR" in analysis_df.columns:

    analysis_df["confidence"] = np.maximum(
        analysis_df["probability_DR"].astype(float),
        1.0 - analysis_df["probability_DR"].astype(float)
    )

    confidence_column = "confidence"

else:

    raise RuntimeError(
        "Neither confidence nor probability_DR exists in analysis_df."
    )


case_definitions = {

    "DS_Correct": (
        (analysis_df["true_class"] == "DS")
        &
        (analysis_df["prediction_status"] == "Correct")
    ),

    "DR_Correct": (
        (analysis_df["true_class"] == "DR")
        &
        (analysis_df["prediction_status"] == "Correct")
    ),

    "DS_Incorrect": (
        (analysis_df["true_class"] == "DS")
        &
        (analysis_df["prediction_status"] == "Incorrect")
    ),

    "DR_Incorrect": (
        (analysis_df["true_class"] == "DR")
        &
        (analysis_df["prediction_status"] == "Incorrect")
    ),

}


selected_cases = []

for group_name, mask in case_definitions.items():

    candidates = analysis_df.loc[
        mask
    ].copy()

    if candidates.empty:
        raise RuntimeError(
            f"No available case for representative group: "
            f"{group_name}"
        )

    candidates = candidates.sort_values(
        by=[
            confidence_column,
            "map_mean",
            "condition_id",
        ],
        ascending=[
            False,
            False,
            True,
        ]
    )

    selected = candidates.iloc[0].copy()

    selected["representative_group"] = (
        group_name
    )

    selected_cases.append(
        selected
    )


representative_df = pd.DataFrame(
    selected_cases
).reset_index(
    drop=True
)

print("\nRepresentative cases selected:")

print(
    representative_df[
        [
            "representative_group",
            "condition_id",
            "true_class",
            "prediction_status",
            confidence_column,
            "map_mean",
        ]
    ].to_string(
        index=False
    )
)


# =============================================================================
# 6. BUILD RECURSIVE FILE INDEX
# =============================================================================

print("\nResolving Grad-CAM files...")


npy_files = list(
    NPY_ROOT.rglob("*.npy")
)

png_files = list(
    PNG_ROOT.rglob("*.png")
)

if len(npy_files) == 0:
    raise RuntimeError(
        "No Grad-CAM NPY files found."
    )

if len(png_files) == 0:
    raise RuntimeError(
        "No Grad-CAM PNG files found."
    )

print(
    "NPY files found:",
    len(npy_files)
)

print(
    "PNG files found:",
    len(png_files)
)


def resolve_file(
    files,
    condition_id
):

    exact_stem = [
        p
        for p in files
        if p.stem == condition_id
    ]

    if len(exact_stem) == 1:
        return exact_stem[0]

    contains = [
        p
        for p in files
        if condition_id in p.stem
    ]

    if len(contains) == 1:
        return contains[0]

    # Handle filenames containing condition ID with extra suffixes.
    normalized = (
        condition_id
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
    )

    normalized_matches = [
        p
        for p in files
        if normalized in p.stem
    ]

    if len(normalized_matches) == 1:
        return normalized_matches[0]

    if len(exact_stem) > 1:
        raise RuntimeError(
            f"Multiple exact files found for {condition_id}."
        )

    if len(contains) > 1:
        raise RuntimeError(
            f"Multiple matching files found for {condition_id}."
        )

    return None


resolved = []

for _, row in representative_df.iterrows():

    condition_id = str(
        row["condition_id"]
    )

    npy_path = resolve_file(
        npy_files,
        condition_id
    )

    png_path = resolve_file(
        png_files,
        condition_id
    )

    if npy_path is None:
        raise FileNotFoundError(
            f"Grad-CAM NPY not found for condition:\n"
            f"{condition_id}"
        )

    if png_path is None:
        raise FileNotFoundError(
            f"Grad-CAM PNG not found for condition:\n"
            f"{condition_id}"
        )

    resolved.append({

        "representative_group":
            row["representative_group"],

        "condition_id":
            condition_id,

        "true_class":
            row["true_class"],

        "prediction_status":
            row["prediction_status"],

        "predicted_class":
            row["predicted_class"]
            if "predicted_class" in row.index
            else "",

        "confidence":
            float(
                row[confidence_column]
            ),

        "map_mean":
            float(
                row["map_mean"]
            ),

        "npy_path":
            npy_path,

        "png_path":
            png_path,

    })


resolved_df = pd.DataFrame(
    resolved
)

print(
    "Representative Grad-CAM files resolved:",
    len(resolved_df),
)


# =============================================================================
# 7. VERIFY MAPS
# =============================================================================

for _, row in resolved_df.iterrows():

    cam = np.load(
        row["npy_path"]
    )

    if cam.shape != (
        224,
        224
    ):
        raise RuntimeError(
            f"Invalid Grad-CAM shape for "
            f"{row['condition_id']}: {cam.shape}"
        )

    if not np.isfinite(
        cam
    ).all():
        raise RuntimeError(
            f"Non-finite Grad-CAM map for "
            f"{row['condition_id']}"
        )


print(
    "Grad-CAM map validation: PASS"
)


# =============================================================================
# 8. CREATE INDIVIDUAL REPRESENTATIVE PANELS
# =============================================================================

def normalize_cam(
    cam
):

    cam = cam.astype(
        np.float32
    )

    minimum = float(
        cam.min()
    )

    maximum = float(
        cam.max()
    )

    if maximum > minimum:

        cam = (
            cam - minimum
        ) / (
            maximum - minimum
        )

    else:

        cam = np.zeros_like(
            cam
        )

    return cam


def create_panel(
    row,
    output_path
):

    cam = np.load(
        row["npy_path"]
    )

    cam = normalize_cam(
        cam
    )

    overlay_image = Image.open(
        row["png_path"]
    ).convert(
        "RGB"
    )

    if overlay_image.size != (
        224,
        224
    ):

        overlay_image = overlay_image.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

    overlay = np.asarray(
        overlay_image,
        dtype=np.float32
    ) / 255.0

    # The Cell 2 PNG is the frozen Grad-CAM overlay.
    # We display its luminance as the background reference.
    background = (
        0.299 * overlay[:, :, 0]
        +
        0.587 * overlay[:, :, 1]
        +
        0.114 * overlay[:, :, 2]
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4.5)
    )

    axes[0].imshow(
        background,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[0].set_title(
        "CXR"
    )

    axes[0].axis(
        "off"
    )

    axes[1].imshow(
        cam,
        cmap="jet",
        vmin=0,
        vmax=1
    )

    axes[1].set_title(
        "Grad-CAM"
    )

    axes[1].axis(
        "off"
    )

    axes[2].imshow(
        overlay
    )

    axes[2].set_title(
        "Grad-CAM Overlay"
    )

    axes[2].axis(
        "off"
    )

    title = (
        f"{row['representative_group'].replace('_', ' ')} | "
        f"Condition: {row['condition_id']}\n"
        f"True: {row['true_class']} | "
        f"Predicted: {row['predicted_class']} | "
        f"Confidence: {row['confidence']:.4f}"
    )

    fig.suptitle(
        title,
        fontsize=11
    )

    fig.tight_layout(
        rect=[
            0,
            0,
            1,
            0.88
        ]
    )

    fig.savefig(
        output_path,
        dpi=600,
        format="png",
        bbox_inches="tight"
    )

    plt.close(
        fig
    )


individual_outputs = []

for _, row in resolved_df.iterrows():

    filename = (
        str(
            row["representative_group"]
        )
        +
        "_"
        +
        str(
            row["condition_id"]
        )
        +
        "_GradCAM.png"
    )

    output_path = (
        OUTPUT_ROOT
        / filename
    )

    create_panel(
        row,
        output_path
    )

    individual_outputs.append(
        output_path
    )

    print(
        f"{row['representative_group']:<20} : PASS"
    )


# =============================================================================
# 9. COMBINED FOUR-CASE FIGURE
# =============================================================================

COMBINED_FIGURE = (
    OUTPUT_ROOT
    / "Four_Representative_GradCAM_Cases.png"
)

fig, axes = plt.subplots(
    4,
    3,
    figsize=(12, 15)
)

for i, (_, row) in enumerate(
    resolved_df.iterrows()
):

    cam = normalize_cam(
        np.load(
            row["npy_path"]
        )
    )

    overlay_image = Image.open(
        row["png_path"]
    ).convert(
        "RGB"
    )

    if overlay_image.size != (
        224,
        224
    ):

        overlay_image = overlay_image.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

    overlay = np.asarray(
        overlay_image,
        dtype=np.float32
    ) / 255.0

    background = (
        0.299 * overlay[:, :, 0]
        +
        0.587 * overlay[:, :, 1]
        +
        0.114 * overlay[:, :, 2]
    )

    axes[i, 0].imshow(
        background,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[i, 1].imshow(
        cam,
        cmap="jet",
        vmin=0,
        vmax=1
    )

    axes[i, 2].imshow(
        overlay
    )

    for j in range(3):
        axes[i, j].axis(
            "off"
        )

    axes[i, 0].set_title(
        "CXR"
    )

    axes[i, 1].set_title(
        "Grad-CAM"
    )

    axes[i, 2].set_title(
        "Grad-CAM Overlay"
    )

    fig.text(
        0.02,
        0.875 - i * 0.215,
        (
            f"{row['representative_group'].replace('_', ' ')}\n"
            f"True={row['true_class']} | "
            f"Pred={row['predicted_class']}\n"
            f"Confidence={row['confidence']:.4f}"
        ),
        fontsize=9,
        va="center"
    )


fig.suptitle(
    "Representative CXR Grad-CAM Explainability Cases",
    fontsize=15
)

fig.tight_layout(
    rect=[
        0.08,
        0,
        1,
        0.96
    ]
)

fig.savefig(
    COMBINED_FIGURE,
    dpi=600,
    format="png",
    bbox_inches="tight"
)

plt.close(
    fig
)


# =============================================================================
# 10. SAVE MANIFEST
# =============================================================================

MANIFEST = (
    OUTPUT_ROOT
    / "Representative_GradCAM_Manifest.csv"
)

resolved_df[
    [
        "representative_group",
        "condition_id",
        "true_class",
        "prediction_status",
        "predicted_class",
        "confidence",
        "map_mean",
        "npy_path",
        "png_path",
    ]
].to_csv(
    MANIFEST,
    index=False
)


# =============================================================================
# 11. SAVE SUMMARY
# =============================================================================

SUMMARY = (
    OUTPUT_ROOT
    / "Notebook17_Cell5_Representative_GradCAM_Summary.json"
)

summary = {

    "status":
        "PASS",

    "representative_cases":
        4,

    "groups": [
        "DS_Correct",
        "DR_Correct",
        "DS_Incorrect",
        "DR_Incorrect",
    ],

    "selection_rule":
        "Highest-confidence case within each true-class/prediction-status group",

    "source":
        "Frozen Cell 2 Grad-CAM outputs and Cell 3 analysis_df",

    "training_performed":
        False,

    "fine_tuning_performed":
        False,

    "model_inference_performed":
        False,

    "checkpoint_selection_performed":
        False,

    "threshold_tuning_performed":
        False,

    "test_predictions_modified":
        False,

    "combined_figure":
        str(COMBINED_FIGURE),

    "manifest":
        str(MANIFEST),

}


with open(
    SUMMARY,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 12. FINAL FILE AUDIT
# =============================================================================

for path in individual_outputs:

    if not path.exists():
        raise RuntimeError(
            f"Representative panel missing:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Representative panel is empty:\n{path}"
        )


if not COMBINED_FIGURE.exists():
    raise RuntimeError(
        "Combined figure was not created."
    )

if not MANIFEST.exists():
    raise RuntimeError(
        "Manifest was not created."
    )

if not SUMMARY.exists():
    raise RuntimeError(
        "Summary was not created."
    )


# =============================================================================
# 13. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 90)
print("NOTEBOOK 17 — CELL 5 STATUS: PASS")
print("=" * 90)

print(
    "Representative cases : 4/4"
)

print(
    "DS Correct            : PASS"
)

print(
    "DR Correct            : PASS"
)

print(
    "DS Incorrect          : PASS"
)

print(
    "DR Incorrect          : PASS"
)

print("\nIndividual figures:")

for path in individual_outputs:
    print(path)

print("\nCombined figure:")
print(COMBINED_FIGURE)

print("\nManifest:")
print(MANIFEST)

print("\nSummary:")
print(SUMMARY)

print("\nTraining performed        : NO")
print("Fine-tuning performed     : NO")
print("Model inference           : NO")
print("Checkpoint selection      : NO")
print("Threshold tuning          : NO")
print("Test predictions modified : NO")

print("\nCELL 5 COMPLETE.")
print("Proceed to Notebook 17 — Cell 6.")

NOTEBOOK 17 — CELL 5
REPRESENTATIVE CXR GRAD-CAM VISUALIZATION
Output directory:
C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Cell5_Representative_GradCAM
Cell 3 analysis data : PASS
Records              : 396
Grad-CAM NPY root    : PASS
Grad-CAM PNG root    : PASS

Representative cases selected:
representative_group                         condition_id true_class prediction_status  confidence  map_mean
          DS_Correct 6009e961-99fd-4755-b10d-04daf9afcc3d         DS           Correct    0.811431  0.173485
          DR_Correct 1065a3a5-3ebf-4173-9e03-a8daef09e6ab         DR           Correct    0.956071  0.234003
        DS_Incorrect 1e7c0d31-fb56-41d2-ace2-9a1eb4fb2769         DS         Incorrect    0.957573  0.235309
        DR_Incorrect 78929a9e-fac9-400c-bb34-2de025e2c89d         DR         Incorrect    0.854052  0.109585

Resolving Grad-CAM files...
NPY files found: 396
PNG files found: 396
Representative Gr

In [19]:
# =============================================================================
# NOTEBOOK 17 — CELL 6
# FINAL XAI SUMMARY + COMPLETE NOTEBOOK 17 AUDIT
# FINAL CORRECTED VERSION
#
# Uses the ACTUAL Cell 2 schema:
# condition_id
# target_binary
# true_class
# official_predicted_binary
# recomputed_predicted_binary
# official_probability_DR
# recomputed_probability_DR
# probability_absolute_difference
# gradcam_mean
# gradcam_max
# gradcam_fraction_ge_0_5
# npy_path
# gradcam_npy_path
# gradcam_png_path
#
# No training / inference / tuning / checkpoint selection.
# =============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu


# =============================================================================
# 1. PATHS
# =============================================================================

FINAL_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT"
)

NOTEBOOK17_ROOT = (
    FINAL_ROOT / "Notebook17_Final_Explainability_Analysis"
)

CELL2_ROOT = (
    NOTEBOOK17_ROOT / "Cell2_CXR_GradCAM"
)

CELL2_CSV = (
    CELL2_ROOT /
    "Notebook17_Cell2_FINAL_396_CXR_GradCAM_Results.csv"
)

NPY_ROOT = (
    CELL2_ROOT / "GradCAM_Maps_NPY"
)

CELL4_ROOT = (
    FINAL_ROOT / "Notebook17_Cell4_GradCAM_Statistical_Analysis"
)

CELL4_TABLE = (
    CELL4_ROOT /
    "Notebook17_Cell4_FINAL_GradCAM_Statistical_Analysis_Table.csv"
)

CELL5_ROOT = (
    FINAL_ROOT / "Notebook17_Cell5_Representative_GradCAM"
)

CELL5_MANIFEST = (
    CELL5_ROOT /
    "Representative_GradCAM_Manifest.csv"
)

CELL5_COMBINED = (
    CELL5_ROOT /
    "Four_Representative_GradCAM_Cases.png"
)

CELL6_ROOT = (
    FINAL_ROOT / "Notebook17_Cell6_Final_XAI_Summary"
)

CELL6_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. START
# =============================================================================

print("=" * 90)
print("NOTEBOOK 17 — CELL 6")
print("FINAL XAI SUMMARY + COMPLETE NOTEBOOK 17 AUDIT")
print("=" * 90)

print("Output directory:")
print(CELL6_ROOT)


# =============================================================================
# 3. INPUT AUDIT
# =============================================================================

required_inputs = {

    "Cell 2 Grad-CAM results":
        CELL2_CSV,

    "Cell 2 NPY directory":
        NPY_ROOT,

    "Cell 4 statistical table":
        CELL4_TABLE,

    "Cell 5 representative manifest":
        CELL5_MANIFEST,

    "Cell 5 combined figure":
        CELL5_COMBINED,

}

print("\n" + "=" * 90)
print("INPUT AUDIT")
print("=" * 90)

for name, path in required_inputs.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<35}: PASS"
    )


# =============================================================================
# 4. LOAD ACTUAL CELL 2 RESULTS
# =============================================================================

gradcam_df = pd.read_csv(
    CELL2_CSV
)

if len(gradcam_df) != 396:

    raise RuntimeError(
        f"Expected 396 Cell 2 records, "
        f"found {len(gradcam_df)}."
    )

if gradcam_df["condition_id"].nunique() != 396:

    raise RuntimeError(
        "Cell 2 condition IDs are not unique."
    )

print("\nActual Cell 2 columns:")
print(
    gradcam_df.columns.tolist()
)


# =============================================================================
# 5. USE THE ACTUAL CELL 2 COLUMN NAMES
# =============================================================================

REQUIRED_COLUMNS = [

    "condition_id",
    "target_binary",
    "true_class",
    "official_predicted_binary",
    "recomputed_predicted_binary",
    "official_probability_DR",
    "recomputed_probability_DR",
    "probability_absolute_difference",
    "gradcam_mean",
    "gradcam_max",
    "gradcam_fraction_ge_0_5",
    "npy_path",
    "gradcam_npy_path",
    "gradcam_png_path",

]

missing = [
    c
    for c in REQUIRED_COLUMNS
    if c not in gradcam_df.columns
]

if missing:

    raise RuntimeError(
        "Actual Cell 2 output is missing required columns:\n"
        +
        "\n".join(missing)
    )

print(
    "\nCell 2 schema audit : PASS"
)


# =============================================================================
# 6. FROZEN PREDICTION CONSISTENCY
# =============================================================================

official_pred = pd.to_numeric(
    gradcam_df[
        "official_predicted_binary"
    ],
    errors="coerce"
)

recomputed_pred = pd.to_numeric(
    gradcam_df[
        "recomputed_predicted_binary"
    ],
    errors="coerce"
)

official_prob = pd.to_numeric(
    gradcam_df[
        "official_probability_DR"
    ],
    errors="coerce"
)

recomputed_prob = pd.to_numeric(
    gradcam_df[
        "recomputed_probability_DR"
    ],
    errors="coerce"
)

if official_pred.isna().any():

    raise RuntimeError(
        "Invalid values found in official_predicted_binary."
    )

if recomputed_pred.isna().any():

    raise RuntimeError(
        "Invalid values found in recomputed_predicted_binary."
    )

if official_prob.isna().any():

    raise RuntimeError(
        "Invalid values found in official_probability_DR."
    )

if recomputed_prob.isna().any():

    raise RuntimeError(
        "Invalid values found in recomputed_probability_DR."
    )

prediction_agreement = int(
    (
        official_pred.values
        ==
        recomputed_pred.values
    ).sum()
)

maximum_probability_difference = float(
    np.max(
        np.abs(
            official_prob.values
            -
            recomputed_prob.values
        )
    )
)

if prediction_agreement != 396:

    raise RuntimeError(
        f"Official/recomputed prediction agreement failed: "
        f"{prediction_agreement}/396"
    )

print(
    "Prediction consistency : PASS"
)

print(
    "Prediction agreement   : 396/396"
)

print(
    f"Maximum probability difference : "
    f"{maximum_probability_difference:.10f}"
)


# =============================================================================
# 7. NORMALIZE TRUE CLASS
# =============================================================================

def normalize_class(value):

    value_str = str(
        value
    ).strip().upper()

    if value_str in {
        "DS",
        "0",
        "SENSITIVE",
        "SENSITIVE-TB",
    }:
        return "DS"

    if value_str in {
        "DR",
        "1",
        "DR-TB",
        "DRUG-RESISTANT",
        "RESISTANT",
    }:
        return "DR"

    raise RuntimeError(
        f"Unknown true_class value: {value}"
    )


gradcam_df["true_class_normalized"] = (
    gradcam_df[
        "true_class"
    ].apply(
        normalize_class
    )
)

gradcam_df["official_predicted_binary_numeric"] = (
    official_pred.astype(int)
)

gradcam_df["prediction_status"] = np.where(
    gradcam_df[
        "target_binary"
    ].astype(int).values
    ==
    gradcam_df[
        "official_predicted_binary_numeric"
    ].values,
    "Correct",
    "Incorrect"
)


# =============================================================================
# 8. TEST COMPOSITION AUDIT
# =============================================================================

ds_count = int(
    (
        gradcam_df[
            "true_class_normalized"
        ]
        == "DS"
    ).sum()
)

dr_count = int(
    (
        gradcam_df[
            "true_class_normalized"
        ]
        == "DR"
    ).sum()
)

correct_count = int(
    (
        gradcam_df[
            "prediction_status"
        ]
        == "Correct"
    ).sum()
)

incorrect_count = int(
    (
        gradcam_df[
            "prediction_status"
        ]
        == "Incorrect"
    ).sum()
)

correct_ds = int(
    (
        (
            gradcam_df[
                "true_class_normalized"
            ]
            == "DS"
        )
        &
        (
            gradcam_df[
                "prediction_status"
            ]
            == "Correct"
        )
    ).sum()
)

correct_dr = int(
    (
        (
            gradcam_df[
                "true_class_normalized"
            ]
            == "DR"
        )
        &
        (
            gradcam_df[
                "prediction_status"
            ]
            == "Correct"
        )
    ).sum()
)

incorrect_ds = int(
    (
        (
            gradcam_df[
                "true_class_normalized"
            ]
            == "DS"
        )
        &
        (
            gradcam_df[
                "prediction_status"
            ]
            == "Incorrect"
        )
    ).sum()
)

incorrect_dr = int(
    (
        (
            gradcam_df[
                "true_class_normalized"
            ]
            == "DR"
        )
        &
        (
            gradcam_df[
                "prediction_status"
            ]
            == "Incorrect"
        )
    ).sum()
)

expected_counts = {

    "DS": 141,
    "DR": 255,
    "Correct": 274,
    "Incorrect": 122,
    "Correct DS": 58,
    "Correct DR": 216,
    "Incorrect DS": 83,
    "Incorrect DR": 39,

}

actual_counts = {

    "DS": ds_count,
    "DR": dr_count,
    "Correct": correct_count,
    "Incorrect": incorrect_count,
    "Correct DS": correct_ds,
    "Correct DR": correct_dr,
    "Incorrect DS": incorrect_ds,
    "Incorrect DR": incorrect_dr,

}

if actual_counts != expected_counts:

    raise RuntimeError(
        "Frozen test composition mismatch.\n"
        f"Expected: {expected_counts}\n"
        f"Actual:   {actual_counts}"
    )

print("\n" + "=" * 90)
print("FROZEN TEST COMPOSITION")
print("=" * 90)

print("Total test cases       :", len(gradcam_df))
print("DS cases               :", ds_count)
print("DR cases               :", dr_count)
print("Correct predictions    :", correct_count)
print("Incorrect predictions  :", incorrect_count)
print("Correct DS             :", correct_ds)
print("Correct DR             :", correct_dr)
print("Incorrect DS           :", incorrect_ds)
print("Incorrect DR           :", incorrect_dr)
print("Composition audit      : PASS")


# =============================================================================
# 9. USE THE ACTUAL CELL 2 GRAD-CAM STATISTICS
# =============================================================================

# Cell 2 provides:
#   gradcam_mean
#   gradcam_max
#   gradcam_fraction_ge_0_5
#
# For ≥0.25, ≥0.75 and ≥0.90, calculate directly from the already-generated
# frozen NPY maps. No model inference is performed.

gradcam_df["gradcam_mean_numeric"] = pd.to_numeric(
    gradcam_df["gradcam_mean"],
    errors="coerce"
)

gradcam_df["gradcam_max_numeric"] = pd.to_numeric(
    gradcam_df["gradcam_max"],
    errors="coerce"
)

gradcam_df["gradcam_fraction_ge_0_5_numeric"] = pd.to_numeric(
    gradcam_df["gradcam_fraction_ge_0_5"],
    errors="coerce"
)

if (
    gradcam_df[
        [
            "gradcam_mean_numeric",
            "gradcam_max_numeric",
            "gradcam_fraction_ge_0_5_numeric",
        ]
    ]
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "Invalid Grad-CAM statistics found in Cell 2 results."
    )


# =============================================================================
# 10. LOAD AND VERIFY ALL 396 NPY MAPS
# =============================================================================

npy_files = list(
    NPY_ROOT.rglob("*.npy")
)

if len(npy_files) != 396:

    raise RuntimeError(
        f"Expected 396 Grad-CAM NPY files, "
        f"found {len(npy_files)}."
    )


npy_index = {}

for path in npy_files:

    npy_index.setdefault(
        path.stem,
        []
    ).append(path)


def resolve_npy(
    condition_id
):

    condition_id = str(
        condition_id
    ).strip()

    if condition_id in npy_index:

        matches = npy_index[
            condition_id
        ]

        if len(matches) == 1:
            return matches[0]

    matches = [
        p
        for p in npy_files
        if condition_id in p.stem
    ]

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve Grad-CAM map for "
        f"{condition_id}. Matches: {len(matches)}"
    )


map_records = []

for _, row in gradcam_df.iterrows():

    condition_id = str(
        row["condition_id"]
    ).strip()

    npy_path = resolve_npy(
        condition_id
    )

    cam = np.load(
        npy_path
    ).astype(
        np.float32
    )

    if cam.shape != (
        224,
        224
    ):

        raise RuntimeError(
            f"Invalid Grad-CAM shape for "
            f"{condition_id}: {cam.shape}"
        )

    if not np.isfinite(
        cam
    ).all():

        raise RuntimeError(
            f"Non-finite Grad-CAM map for "
            f"{condition_id}"
        )

    cam_min = float(
        cam.min()
    )

    cam_max = float(
        cam.max()
    )

    if cam_max > cam_min:

        cam_normalized = (
            cam - cam_min
        ) / (
            cam_max - cam_min
        )

    else:

        cam_normalized = np.zeros_like(
            cam
        )

    map_records.append({

        "condition_id":
            condition_id,

        "true_class":
            row[
                "true_class_normalized"
            ],

        "prediction_status":
            row[
                "prediction_status"
            ],

        "mean_activation":
            float(
                cam_normalized.mean()
            ),

        "max_activation":
            float(
                cam_normalized.max()
            ),

        "fraction_ge_0_25":
            float(
                (
                    cam_normalized
                    >=
                    0.25
                ).mean()
            ),

        "fraction_ge_0_50":
            float(
                (
                    cam_normalized
                    >=
                    0.50
                ).mean()
            ),

        "fraction_ge_0_75":
            float(
                (
                    cam_normalized
                    >=
                    0.75
                ).mean()
            ),

        "fraction_ge_0_90":
            float(
                (
                    cam_normalized
                    >=
                    0.90
                ).mean()
            ),

        "npy_path":
            str(npy_path),

    })


analysis_df = pd.DataFrame(
    map_records
)

if len(analysis_df) != 396:

    raise RuntimeError(
        "Grad-CAM map analysis did not produce 396 records."
    )

print("\n" + "=" * 90)
print("GRADCAM MAP AUDIT")
print("=" * 90)

print("Grad-CAM maps verified    : 396/396")
print("Map shape                 : 224 × 224")
print("Finite-value audit        : PASS")


# =============================================================================
# 11. VERIFY CELL 2 STORED GRAD-CAM VALUES
# =============================================================================

merged_check = gradcam_df[
    [
        "condition_id",
        "gradcam_mean_numeric",
        "gradcam_max_numeric",
        "gradcam_fraction_ge_0_5_numeric",
    ]
].merge(
    analysis_df[
        [
            "condition_id",
            "mean_activation",
            "max_activation",
            "fraction_ge_0_50",
        ]
    ],
    on="condition_id",
    how="inner"
)

if len(merged_check) != 396:

    raise RuntimeError(
        "Cell 2 Grad-CAM records could not be aligned with NPY maps."
    )

mean_diff = float(
    np.max(
        np.abs(
            merged_check[
                "gradcam_mean_numeric"
            ].values
            -
            merged_check[
                "mean_activation"
            ].values
        )
    )
)

max_diff = float(
    np.max(
        np.abs(
            merged_check[
                "gradcam_max_numeric"
            ].values
            -
            merged_check[
                "max_activation"
            ].values
        )
    )
)

area50_diff = float(
    np.max(
        np.abs(
            merged_check[
                "gradcam_fraction_ge_0_5_numeric"
            ].values
            -
            merged_check[
                "fraction_ge_0_50"
            ].values
        )
    )
)

if mean_diff > 1e-5:
    raise RuntimeError(
        f"Grad-CAM mean consistency failed: {mean_diff}"
    )

if max_diff > 1e-5:
    raise RuntimeError(
        f"Grad-CAM max consistency failed: {max_diff}"
    )

if area50_diff > 1e-5:
    raise RuntimeError(
        f"Grad-CAM area ≥0.50 consistency failed: {area50_diff}"
    )

print(
    "Cell 2 ↔ NPY statistic consistency : PASS"
)


# =============================================================================
# 12. OVERALL GRAD-CAM STATISTICS
# =============================================================================

mean_activation = float(
    analysis_df[
        "mean_activation"
    ].mean()
)

median_activation = float(
    analysis_df[
        "mean_activation"
    ].median()
)

mean_area_025 = float(
    analysis_df[
        "fraction_ge_0_25"
    ].mean()
)

mean_area_050 = float(
    analysis_df[
        "fraction_ge_0_50"
    ].mean()
)

mean_area_075 = float(
    analysis_df[
        "fraction_ge_0_75"
    ].mean()
)

mean_area_090 = float(
    analysis_df[
        "fraction_ge_0_90"
    ].mean()
)


# =============================================================================
# 13. DS VS DR STATISTICAL ANALYSIS
# =============================================================================

metrics = [

    "mean_activation",
    "max_activation",
    "fraction_ge_0_25",
    "fraction_ge_0_50",
    "fraction_ge_0_75",
    "fraction_ge_0_90",

]


def bh_fdr(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    n = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted = np.empty(
        n,
        dtype=float
    )

    running = 1.0

    for i in range(
        n - 1,
        -1,
        -1
    ):

        rank = i + 1

        value = (
            ranked[i]
            *
            n
            /
            rank
        )

        running = min(
            running,
            value
        )

        adjusted[i] = running

    result = np.empty(
        n,
        dtype=float
    )

    result[
        order
    ] = np.clip(
        adjusted,
        0,
        1
    )

    return result


ds_df = analysis_df.loc[
    analysis_df["true_class"] == "DS"
]

dr_df = analysis_df.loc[
    analysis_df["true_class"] == "DR"
]

ds_dr_rows = []
ds_dr_pvalues = []

for metric in metrics:

    x = ds_df[
        metric
    ].to_numpy(
        dtype=float
    )

    y = dr_df[
        metric
    ].to_numpy(
        dtype=float
    )

    if (
        np.all(x == x[0])
        and
        np.all(y == y[0])
    ):

        u_value = (
            len(x)
            *
            len(y)
            /
            2
        )

        p_value = 1.0

    else:

        result = mannwhitneyu(
            x,
            y,
            alternative="two-sided"
        )

        u_value = float(
            result.statistic
        )

        p_value = float(
            result.pvalue
        )

    ds_dr_pvalues.append(
        p_value
    )

    pooled_sd = np.sqrt(
        (
            (
                (len(x) - 1)
                *
                np.var(
                    x,
                    ddof=1
                )
            )
            +
            (
                (len(y) - 1)
                *
                np.var(
                    y,
                    ddof=1
                )
            )
        )
        /
        (
            len(x)
            +
            len(y)
            -
            2
        )
    )

    cohens_d = (
        (
            x.mean()
            -
            y.mean()
        )
        /
        pooled_sd
        if pooled_sd > 0
        else 0.0
    )

    ds_dr_rows.append({

        "comparison":
            "DS vs DR",

        "variable":
            metric,

        "n_group_1":
            len(x),

        "n_group_2":
            len(y),

        "mean_group_1":
            float(x.mean()),

        "mean_group_2":
            float(y.mean()),

        "median_group_1":
            float(np.median(x)),

        "median_group_2":
            float(np.median(y)),

        "mann_whitney_U":
            float(u_value),

        "mann_whitney_p":
            p_value,

        "cohens_d":
            float(cohens_d),

    })


ds_dr_fdr = bh_fdr(
    ds_dr_pvalues
)

for i, value in enumerate(
    ds_dr_fdr
):

    ds_dr_rows[i][
        "mann_whitney_p_fdr"
    ] = float(value)

    ds_dr_rows[i][
        "fdr_significant_alpha_0_05"
    ] = bool(
        value < 0.05
    )


ds_dr_stats = pd.DataFrame(
    ds_dr_rows
)


# =============================================================================
# 14. CORRECT VS INCORRECT STATISTICAL ANALYSIS
# =============================================================================

correct_df = analysis_df.loc[
    analysis_df[
        "prediction_status"
    ]
    ==
    "Correct"
]

incorrect_df = analysis_df.loc[
    analysis_df[
        "prediction_status"
    ]
    ==
    "Incorrect"
]

correct_rows = []
correct_pvalues = []

for metric in metrics:

    x = correct_df[
        metric
    ].to_numpy(
        dtype=float
    )

    y = incorrect_df[
        metric
    ].to_numpy(
        dtype=float
    )

    if (
        np.all(x == x[0])
        and
        np.all(y == y[0])
    ):

        u_value = (
            len(x)
            *
            len(y)
            /
            2
        )

        p_value = 1.0

    else:

        result = mannwhitneyu(
            x,
            y,
            alternative="two-sided"
        )

        u_value = float(
            result.statistic
        )

        p_value = float(
            result.pvalue
        )

    correct_pvalues.append(
        p_value
    )

    pooled_sd = np.sqrt(
        (
            (
                (len(x) - 1)
                *
                np.var(
                    x,
                    ddof=1
                )
            )
            +
            (
                (len(y) - 1)
                *
                np.var(
                    y,
                    ddof=1
                )
            )
        )
        /
        (
            len(x)
            +
            len(y)
            -
            2
        )
    )

    cohens_d = (
        (
            x.mean()
            -
            y.mean()
        )
        /
        pooled_sd
        if pooled_sd > 0
        else 0.0
    )

    correct_rows.append({

        "comparison":
            "Correct vs Incorrect",

        "variable":
            metric,

        "n_group_1":
            len(x),

        "n_group_2":
            len(y),

        "mean_group_1":
            float(x.mean()),

        "mean_group_2":
            float(y.mean()),

        "median_group_1":
            float(np.median(x)),

        "median_group_2":
            float(np.median(y)),

        "mann_whitney_U":
            float(u_value),

        "mann_whitney_p":
            p_value,

        "cohens_d":
            float(cohens_d),

    })


correct_fdr = bh_fdr(
    correct_pvalues
)

for i, value in enumerate(
    correct_fdr
):

    correct_rows[i][
        "mann_whitney_p_fdr"
    ] = float(value)

    correct_rows[i][
        "fdr_significant_alpha_0_05"
    ] = bool(
        value < 0.05
    )


correctness_stats = pd.DataFrame(
    correct_rows
)


# =============================================================================
# 15. SIGNIFICANCE COUNTS
# =============================================================================

ds_dr_significant = int(
    ds_dr_stats[
        "fdr_significant_alpha_0_05"
    ].sum()
)

correctness_significant = int(
    correctness_stats[
        "fdr_significant_alpha_0_05"
    ].sum()
)


# =============================================================================
# 16. CONFIDENCE CORRELATIONS
# =============================================================================

confidence_correlations = {}

if "official_probability_DR" in gradcam_df.columns:

    confidence = np.maximum(
        official_prob.values,
        1.0 - official_prob.values
    )

    correlation_df = pd.DataFrame({

        "confidence":
            confidence,

        "mean_activation":
            analysis_df[
                "mean_activation"
            ].values,

        "fraction_ge_0_50":
            analysis_df[
                "fraction_ge_0_50"
            ].values,

        "fraction_ge_0_75":
            analysis_df[
                "fraction_ge_0_75"
            ].values,

        "fraction_ge_0_90":
            analysis_df[
                "fraction_ge_0_90"
            ].values,

    })

    for metric in [

        "mean_activation",
        "fraction_ge_0_50",
        "fraction_ge_0_75",
        "fraction_ge_0_90",

    ]:

        confidence_correlations[
            f"confidence_vs_{metric}_spearman"
        ] = float(
            correlation_df[
                [
                    "confidence",
                    metric
                ]
            ].corr(
                method="spearman"
            ).iloc[0, 1]
        )


# =============================================================================
# 17. CELL 5 REPRESENTATIVE CASE AUDIT
# =============================================================================

representative_df = pd.read_csv(
    CELL5_MANIFEST
)

if len(representative_df) != 4:

    raise RuntimeError(
        f"Expected 4 representative cases, "
        f"found {len(representative_df)}."
    )

expected_groups = {

    "DS_Correct",
    "DR_Correct",
    "DS_Incorrect",
    "DR_Incorrect",

}

actual_groups = set(
    representative_df[
        "representative_group"
    ].astype(str)
)

if actual_groups != expected_groups:

    raise RuntimeError(
        "Cell 5 representative groups mismatch."
    )

print(
    "\nRepresentative cases : PASS (4/4)"
)


# =============================================================================
# 18. SAVE STATISTICAL TABLE
# =============================================================================

STAT_TABLE = (
    CELL6_ROOT
    / "Notebook17_Cell6_FINAL_GradCAM_Statistical_Table.csv"
)

combined_stats = pd.concat(
    [
        ds_dr_stats,
        correctness_stats,
    ],
    ignore_index=True
)

combined_stats.to_csv(
    STAT_TABLE,
    index=False
)


# =============================================================================
# 19. SAVE SUMMARY TABLE
# =============================================================================

SUMMARY_TABLE = (
    CELL6_ROOT
    / "Notebook17_Cell6_FINAL_XAI_Summary_Table.csv"
)

summary_table = pd.DataFrame({

    "Analysis": [

        "Grad-CAM maps analyzed",
        "DS-TB cases",
        "DR-TB cases",
        "Correct predictions",
        "Incorrect predictions",
        "Correct DS predictions",
        "Correct DR predictions",
        "Incorrect DS predictions",
        "Incorrect DR predictions",
        "Mean Grad-CAM activation",
        "Median Grad-CAM activation",
        "Mean area ≥ 0.25",
        "Mean area ≥ 0.50",
        "Mean area ≥ 0.75",
        "Mean area ≥ 0.90",
        "DS-vs-DR FDR-significant measures",
        "Correct-vs-incorrect FDR-significant measures",

    ],

    "Value": [

        396,
        141,
        255,
        274,
        122,
        58,
        216,
        83,
        39,
        mean_activation,
        median_activation,
        mean_area_025,
        mean_area_050,
        mean_area_075,
        mean_area_090,
        ds_dr_significant,
        correctness_significant,

    ],

})

summary_table.to_csv(
    SUMMARY_TABLE,
    index=False
)


# =============================================================================
# 20. COMPLETE AUDIT
# =============================================================================

AUDIT_FILE = (
    CELL6_ROOT
    / "Notebook17_Cell6_COMPLETE_XAI_Audit.csv"
)

audit_df = pd.DataFrame({

    "Component": [

        "Cell 1 — XAI Safety + Frozen Model Audit",
        "Cell 2 — Frozen CoAtNet CXR Grad-CAM",
        "Cell 3 — Grad-CAM Quantitative Analysis",
        "Cell 3 — Publication Figures",
        "Cell 4 — Grad-CAM Statistical Analysis",
        "Cell 5 — Representative Grad-CAM Visualization",
        "Grad-CAM test cases",
        "Grad-CAM maps",
        "Grad-CAM map shape",
        "Official/recomputed prediction agreement",
        "Training",
        "Fine-tuning",
        "Model inference in Cell 6",
        "Checkpoint selection",
        "Threshold tuning",
        "Test predictions modified",

    ],

    "Status": [

        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "396",
        "396/396",
        "224 × 224",
        "396/396",
        "NO",
        "NO",
        "NO",
        "NO",
        "NO",
        "NO",

    ],

})

audit_df.to_csv(
    AUDIT_FILE,
    index=False
)


# =============================================================================
# 21. FINAL JSON
# =============================================================================

FINAL_JSON = (
    CELL6_ROOT
    / "Notebook17_Cell6_FINAL_XAI_Summary.json"
)

final_summary = {

    "status":
        "PASS",

    "test_cases":
        396,

    "test_DS":
        141,

    "test_DR":
        255,

    "correct_predictions":
        274,

    "incorrect_predictions":
        122,

    "correct_DS":
        58,

    "correct_DR":
        216,

    "incorrect_DS":
        83,

    "incorrect_DR":
        39,

    "gradcam_maps":
        396,

    "gradcam_shape":
        [224, 224],

    "official_recomputed_prediction_agreement":
        "396/396",

    "maximum_probability_difference":
        maximum_probability_difference,

    "mean_activation":
        mean_activation,

    "median_activation":
        median_activation,

    "mean_fraction_ge_0_25":
        mean_area_025,

    "mean_fraction_ge_0_50":
        mean_area_050,

    "mean_fraction_ge_0_75":
        mean_area_075,

    "mean_fraction_ge_0_90":
        mean_area_090,

    "DS_vs_DR_FDR_significant_measures":
        ds_dr_significant,

    "Correct_vs_Incorrect_FDR_significant_measures":
        correctness_significant,

    "confidence_spearman_correlations":
        confidence_correlations,

    "representative_cases":
        representative_df[
            [
                "representative_group",
                "condition_id",
                "true_class",
                "prediction_status",
            ]
        ].to_dict(
            orient="records"
        ),

    "scientific_interpretation": {

        "DS_vs_DR":
            "No tested Grad-CAM quantitative measure showed "
            "a statistically significant DS-vs-DR difference "
            "after Benjamini-Hochberg FDR correction.",

        "Correct_vs_Incorrect":
            "No tested Grad-CAM quantitative measure showed "
            "a statistically significant difference between "
            "correct and incorrect predictions after "
            "Benjamini-Hochberg FDR correction.",

        "interpretation_scope":
            "Grad-CAM provides attribution evidence for the "
            "frozen CXR-only CoAtNet model. The quantitative "
            "analysis does not establish statistically significant "
            "class-specific localization differences.",

    },

    "training_performed":
        False,

    "fine_tuning_performed":
        False,

    "model_inference_performed":
        False,

    "checkpoint_selection_performed":
        False,

    "threshold_tuning_performed":
        False,

    "test_predictions_modified":
        False,

    "outputs": {

        "statistical_table":
            str(STAT_TABLE),

        "summary_table":
            str(SUMMARY_TABLE),

        "audit":
            str(AUDIT_FILE),

        "final_summary":
            str(FINAL_JSON),

        "representative_figure":
            str(CELL5_COMBINED),

    },

}

with open(
    FINAL_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_summary,
        f,
        indent=4
    )


# =============================================================================
# 22. FINAL OUTPUT VERIFICATION
# =============================================================================

for path in [

    STAT_TABLE,
    SUMMARY_TABLE,
    AUDIT_FILE,
    FINAL_JSON,

]:

    if not path.exists():

        raise RuntimeError(
            f"Final output missing:\n{path}"
        )

    if path.stat().st_size == 0:

        raise RuntimeError(
            f"Final output is empty:\n{path}"
        )


# =============================================================================
# 23. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 90)
print("NOTEBOOK 17 — CELL 6 STATUS: PASS")
print("=" * 90)

print(
    "Grad-CAM cases analyzed     : 396"
)

print(
    "DS cases                    : 141"
)

print(
    "DR cases                    : 255"
)

print(
    "Correct predictions         : 274"
)

print(
    "Incorrect predictions       : 122"
)

print(
    "Grad-CAM maps               : 396/396"
)

print(
    "Map shape                   : 224 × 224"
)

print(
    "Prediction agreement       : 396/396"
)

print(
    "DS-vs-DR FDR significant    :",
    ds_dr_significant
)

print(
    "Correct-vs-incorrect FDR significant :",
    correctness_significant
)

print(
    "\nTraining performed         : NO"
)

print(
    "Fine-tuning performed      : NO"
)

print(
    "Model inference performed  : NO"
)

print(
    "Checkpoint selection       : NO"
)

print(
    "Threshold tuning           : NO"
)

print(
    "Test predictions modified  : NO"
)

print("\nStatistical table:")
print(STAT_TABLE)

print("\nSummary table:")
print(SUMMARY_TABLE)

print("\nComplete audit:")
print(AUDIT_FILE)

print("\nFinal summary:")
print(FINAL_JSON)

print("\n" + "=" * 90)
print("NOTEBOOK 17 COMPLETE")
print("=" * 90)

NOTEBOOK 17 — CELL 6
FINAL XAI SUMMARY + COMPLETE NOTEBOOK 17 AUDIT
Output directory:
C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\Notebook17_Cell6_Final_XAI_Summary

INPUT AUDIT
Cell 2 Grad-CAM results            : PASS
Cell 2 NPY directory               : PASS
Cell 4 statistical table           : PASS
Cell 5 representative manifest     : PASS
Cell 5 combined figure             : PASS

Actual Cell 2 columns:
['condition_id', 'target_binary', 'true_class', 'official_predicted_binary', 'recomputed_predicted_binary', 'official_probability_DR', 'recomputed_probability_DR', 'probability_absolute_difference', 'gradcam_mean', 'gradcam_max', 'gradcam_fraction_ge_0_5', 'npy_path', 'gradcam_npy_path', 'gradcam_png_path']

Cell 2 schema audit : PASS
Prediction consistency : PASS
Prediction agreement   : 396/396
Maximum probability difference : 0.0000016689

FROZEN TEST COMPOSITION
Total test cases       : 396
DS cases               : 141
